In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, SubsetRandomSampler

import snntorch as snn
from snntorch import spikegen, utils, surrogate
from snntorch.functional import quant

import brevitas.nn as qnn
# from brevitas.nn import QuantConv1d, QuantIdentity, QuantLinear, BatchNorm1dToQuantScaleBias
from brevitas.quant import Int8WeightPerTensorFloat, Int8ActPerTensorFloat, Int32Bias
from brevitas.export import export_qonnx
from brevitas.core.scaling      import ScalingImplType
from brevitas.core.restrict_val import RestrictValueType
from brevitas.core.quant        import QuantType


from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
#from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

import optuna

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import itertools
import csv, copy, glob
import imblearn, imblearn.over_sampling
from collections import Counter, OrderedDict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve, auc

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import label_binarize, normalize
from scipy.signal import butter, lfilter, freqz

import os, sys, time, datetime, argparse, json


/home/velox-217533/anaconda3/envs/fau_snn_torch-cuda/lib/python3.12/site-packages/brevitas/graph/equalize.py:69: UserWarning: fast_hadamard_transform package not found, using standard pytorch kernels
  warnings.warn("fast_hadamard_transform package not found, using standard pytorch kernels")
/home/velox-217533/anaconda3/envs/fau_snn_torch-cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
# random.seed(seed)
torch.cuda.manual_seed_all(seed) # for multi-GPU setups

In [3]:
# def normalize_row_to_range(row, low=-4.0, high=4.0):
#     rmin, rmax = row.min(), row.max()
#     denom = (rmax - rmin) if rmax > rmin else 1e-8
#     return low + (row - rmin) * (high - low) / denom

In [4]:
import os, glob
import numpy as np
import pandas as pd
import torch
from collections import Counter

def load_train_test_data(
    mitbih_path='/home/velox-217533/Projects/fau_projects/research/data/mitbih_processed_intra_patient_4class_180_center90_rr_features_filtered/train',
    folders=None,
    random_state=42,
    max_files_per_folder=None,
    rr_cols=4,
    beat_len=180,
    rr_stats=None,              # pass rr_stats here for val/test to avoid leakage
    return_rr_stats=False,      # True for train load, False for test load
    eps=1e-8,
):
    if folders is None:
        folders = ['normal', 'sveb', 'veb', 'f']
    label_mapping = {'normal': 0, 'sveb': 1, 'veb': 2, 'f': 3}

    expected_cols = beat_len + rr_cols  # 184

    X_parts, y_parts = [], []

    for folder in folders:
        folder_path = os.path.join(mitbih_path, folder)
        csvs = sorted(
            glob.glob(os.path.join(folder_path, '*.csv')) +
            glob.glob(os.path.join(folder_path, '*.CSV'))
        )
        if not csvs:
            print(f'Warning: no CSVs found in: {folder_path}')
            continue

        if max_files_per_folder is not None:
            csvs = csvs[:max_files_per_folder]

        for fpath in csvs:
            df = pd.read_csv(
                fpath, dtype=np.float32, engine='c',
                usecols=lambda c: c != 'Unnamed: 0'
            )
            if df.shape[0] == 0:
                df = pd.read_csv(fpath, header=None, dtype=np.float32, engine='c')

            df = df.dropna(axis=1, how='all')
            arr = df.to_numpy(copy=False)

            if arr.shape[1] != expected_cols:
                raise ValueError(
                    f'Expected {expected_cols} cols (180+4). '
                    f'Got {arr.shape[1]} in {fpath}'
                )

            X_parts.append(arr)
            y_parts.extend([label_mapping[folder]] * arr.shape[0])

    if not X_parts:
        raise FileNotFoundError(f'No usable CSV rows under {mitbih_path}')

    X = np.vstack(X_parts).astype(np.float32, copy=False)
    y = np.asarray(y_parts, dtype=np.int64)

    # shuffle
    rng = np.random.RandomState(random_state)
    perm = rng.permutation(len(y))
    X = X[perm]
    y = y[perm]

    # split beat + rr
    X_beat = X[:, :beat_len]                  # (N,180) already z-scored per beat
    X_rr   = X[:, beat_len:beat_len+rr_cols]  # (N,4)   normalize separately

    # sanitize
    X_beat = np.nan_to_num(X_beat, nan=0.0).astype(np.float32, copy=False)
    X_rr   = np.nan_to_num(X_rr,   nan=0.0).astype(np.float32, copy=False)

    # rr normalization (train stats only)
    if rr_stats is None:
        rr_mean = X_rr.mean(axis=0)
        rr_std  = X_rr.std(axis=0)
        rr_std  = np.maximum(rr_std, eps)
        rr_stats = {"mean": rr_mean.astype(np.float32), "std": rr_std.astype(np.float32)}
    else:
        rr_mean = rr_stats["mean"]
        rr_std  = np.maximum(rr_stats["std"], eps)

    X_rr_norm = (X_rr - rr_mean) / rr_std

    # recombine into 184
    X_final = np.concatenate([X_beat, X_rr_norm], axis=1).astype(np.float32, copy=False)

    # prints separated (NO confusion)
    print(f"Loaded: {mitbih_path}  X={X_final.shape}")
    print(f"Beat(180) stats: min={X_beat.min():.2f} max={X_beat.max():.2f} mean={X_beat.mean():.4f} std={X_beat.std():.4f}")
    print(f"RR(4) raw stats : min={X_rr.min():.2f} max={X_rr.max():.2f} mean={X_rr.mean():.4f} std={X_rr.std():.4f}")
    print(f"RR(4) norm stats: min={X_rr_norm.min():.2f} max={X_rr_norm.max():.2f} mean={X_rr_norm.mean():.4f} std={X_rr_norm.std():.4f}")

    # torch tensors
    X_tensor = torch.from_numpy(X_final).unsqueeze(1)  # (N,1,184)
    y_tensor = torch.from_numpy(y)                     # (N,)

    print("Class distribution:", Counter(y_tensor.tolist()))
    print("X shape:", tuple(X_tensor.shape), "| y shape:", tuple(y_tensor.shape))

    if return_rr_stats:
        return X_tensor, y_tensor, rr_stats
    return X_tensor, y_tensor


In [5]:
import torch
from torch.utils.data import TensorDataset

def create_csnn_datasets(X_train, y_train, X_test=None, y_test=None):
    """
    Create TensorDataset objects for training/evaluation.

    Assumes:
      - X_train is a torch.Tensor of shape (N, 1, L)  (L can be 180 or 184)
      - y_train is a torch.Tensor of shape (N,)
    """
    train_dataset = TensorDataset(X_train, y_train)

    if X_test is None:
        print("train_dataset created successfully")
        return train_dataset

    test_dataset = TensorDataset(X_test, y_test)
    print("train_dataset and test_dataset created successfully")
    return train_dataset, test_dataset


In [6]:
def create_qcsnn_model24_rr_both(num_bits=8, input_size=180, rr_dim=4, stride4=1, kernel_size=3, 
                                  dropout4=0.35, beta4=0.5, slope4=25, 
                                  threshold4=0.5, learn_beta4=True, learn_threshold4=True):
    """Factory function for 4-class QCSNN — RR features routed to BOTH stages"""

    spike_grad4 = snn.surrogate.fast_sigmoid(slope=slope4)
    
    # Calculate output sizes after 3 conv blocks
    output_size1 = (input_size - kernel_size) // stride4 + 1
    output_size1 = output_size1 // 2  # MaxPool
    
    output_size2 = (output_size1 - kernel_size) // stride4 + 1
    output_size2 = output_size2 // 2  # MaxPool
    
    output_size3 = (output_size2 - kernel_size) // stride4 + 1
    output_size3 = output_size3 // 2  # MaxPool
    
    flattened_size = output_size3 * 24  # 480 when input_size=180

    print(f"Output sizes: Block1={output_size1}, Block2={output_size2}, Block3={output_size3}")
    print(f"Flattened size: {flattened_size}")
    print(f"Stage 1 input: {flattened_size + rr_dim} (trunk + RR)")
    print(f"Stage 2 input: {flattened_size + rr_dim} (trunk + RR)")  # <-- CHANGED
    
    # Backbone (unchanged)
    csnet24 = torch.nn.Sequential(OrderedDict([
        ("qcsnet24_cblk1_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
    
        ("qcsnet24_cblk1_qconv1d",
         qnn.QuantConv1d(
             1, 16, 3, stride=stride4, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
    
        ("qcsnet24_cblk1_batch_norm",
         qnn.BatchNorm1dToQuantScaleBias(
             16,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             bias_quant=Int32Bias,
             return_quant_tensor=True)),

        ("qcsnet24_cblk1_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True)),
        ("qcsnet24_cblk1_max_pool", torch.nn.MaxPool1d(2, 2)),
    
        ("qcsnet24_cblk2_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
        
        ("qcsnet24_cblk2_qconv1d",
         qnn.QuantConv1d(
             16, 16, 3, stride=stride4, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
    
        ("qcsnet24_cblk2_batch_norm",
         qnn.BatchNorm1dToQuantScaleBias(
             16,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             bias_quant=Int32Bias,
             return_quant_tensor=True)),
    
        ("qcsnet24_cblk2_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True)),
        ("qcsnet24_cblk2_max_pool", torch.nn.MaxPool1d(2, 2)),
    
        ("qcsnet24_cblk3_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
        
        ("qcsnet24_cblk3_qconv1d",
         qnn.QuantConv1d(
             16, 24, 3, stride=stride4, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
    
        ("qcsnet24_cblk3_batch_norm",
         qnn.BatchNorm1dToQuantScaleBias(
             24,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             bias_quant=Int32Bias,
             return_quant_tensor=True)),
        
        ("qcsnet24_cblk3_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True)),
        ("qcsnet24_cblk3_max_pool", torch.nn.MaxPool1d(2, 2)),
        
        ("qcsnet24_flatten", torch.nn.Flatten()),
    ]))
    
    # Stage 1 head (binary) — 484 input (unchanged)
    csnet2_head = torch.nn.Sequential(OrderedDict([
        ("qcsnet2_lblk1_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),

        ("qcsnet2_lblk1_qlinear",
         qnn.QuantLinear(
             flattened_size + rr_dim,  # 484
             2, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
        ("qcsnet2_lblk1_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True, output=True)),
    ]))

    # Stage 2 head (4-class) — NOW 484 input (CHANGED from 480)
    csnet4_head = torch.nn.Sequential(OrderedDict([
        ("qcsnet4_lblk1_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
        
        ("qcsnet4_lblk1_qlinear",
         qnn.QuantLinear(
             flattened_size + rr_dim,  # <-- CHANGED: 484 instead of 480
             128, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
        ("qcsnet4_lblk1_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True)),
    
        ("qcsnet4_lblk2_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
        
        ("qcsnet4_lblk2_qlinear",
         qnn.QuantLinear(
             128, 4, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
        ("qcsnet4_lblk2_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True, output=True)),
    ]))
    
    # Fix runtime_shape for BatchNorm
    csnet24.qcsnet24_cblk1_batch_norm.runtime_shape = (1, -1, 1)
    csnet24.qcsnet24_cblk2_batch_norm.runtime_shape = (1, -1, 1)
    csnet24.qcsnet24_cblk3_batch_norm.runtime_shape = (1, -1, 1)
    
    model24 = {"net24": csnet24, 
               "net2": csnet2_head,
               "net4": csnet4_head}

    return model24

def forward_pass_rr_both(net, num_steps, data, sig_len=180, rr_dim=4):
    """Forward pass with RR features routed to BOTH stages"""
    net24 = net["net24"]
    net2 = net["net2"]
    net4 = net["net4"]

    mem2_rec, mem4_rec = [], []
    spk2_rec, spk4_rec = [], []

    utils.reset(net24)
    utils.reset(net2)
    utils.reset(net4)

    # data is (B, 1, 184)
    x_sig = data[:, :, :sig_len]                           # (B, 1, 180) -> trunk
    rr = data[:, :, sig_len:sig_len + rr_dim].squeeze(1)   # (B, 4)

    for step in range(num_steps):
        out_body = net24(x_sig)                            # (B, 480)

        # Stage 1: trunk + RR
        out2_in = torch.cat([out_body, rr], dim=1)         # (B, 484)

        # Stage 2: trunk + RR (CHANGED — was just out_body)
        out4_in = torch.cat([out_body, rr], dim=1)         # (B, 484)

        spk_out2, mem_out2 = net2(out2_in)
        spk_out4, mem_out4 = net4(out4_in)                 # <-- uses 484 now

        spk2_rec.append(spk_out2)
        mem2_rec.append(mem_out2)
        spk4_rec.append(spk_out4)
        mem4_rec.append(mem_out4)

    return torch.stack(spk2_rec), torch.stack(mem2_rec), torch.stack(spk4_rec), torch.stack(mem4_rec)

In [7]:
def train_epoch_rr_both(
    model24, dataloader, loss_func, optimizer24, device, num_steps,
    lambda_bin=1.0, lambda_multi=1.0,
):
    """Train one epoch — RR to both stages"""
    train24_loss, train2_correct, train4_correct = 0.0, 0, 0
    model24["net24"].train()
    model24["net2"].train()
    model24["net4"].train()

    for inputs, targets in dataloader:
        optimizer24.zero_grad(set_to_none=True)
        inputs = inputs.to(device, non_blocking=True)
        targets24 = targets.to(device, non_blocking=True).long()
        targets2 = (targets24 > 0).long()

        assert inputs.size(-1) == 184, f"Expected inputs[...,184] but got {tuple(inputs.shape)}"

        spk2, _, spk4, _ = forward_pass_rr_both(model24, num_steps, inputs)
        out2 = spk2.mean(0)
        out4 = spk4.mean(0)

        loss_bin = loss_func(out2, targets2)
        loss_mul = loss_func(out4, targets24)
        loss24 = (lambda_bin * loss_bin) + (lambda_multi * loss_mul)

        loss24.backward()
        optimizer24.step()

        train24_loss += float(loss24.item())
        train2_correct += (out2.argmax(1) == targets2).sum().item()
        train4_correct += (out4.argmax(1) == targets24).sum().item()

    return train24_loss, train2_correct, train4_correct




In [8]:
def validation_epoch_rr_both(model24, dataloader, loss_func, device, num_steps,
                              lambda_bin=1.0, lambda_multi=1.0):
    """Validation epoch — RR to both stages"""
    model24["net24"].eval()
    model24["net2"].eval()
    model24["net4"].eval()

    valid24_loss = 0.0
    valid2_correct, valid4_correct = 0, 0
    n_samples = 0

    tp2 = [0, 0]; fp2 = [0, 0]; tn2 = [0, 0]; fn2 = [0, 0]
    tp4 = [0, 0, 0, 0]; fp4 = [0, 0, 0, 0]; tn4 = [0, 0, 0, 0]; fn4 = [0, 0, 0, 0]

    with torch.no_grad():
        for x, y_multi in dataloader:
            x = x.to(device, non_blocking=True)
            y_multi = y_multi.to(device, non_blocking=True).long()
            y_bin = (y_multi > 0).long()

            assert x.size(-1) == 184, f"Expected x[...,184] but got {tuple(x.shape)}"

            bs = x.size(0)
            n_samples += bs

            spk2, _, spk4, _ = forward_pass_rr_both(model24, num_steps, x)
            out2 = spk2.mean(0)
            out4 = spk4.mean(0)

            loss_bin = loss_func(out2, y_bin)
            loss_mul = loss_func(out4, y_multi)
            loss24 = (lambda_bin * loss_bin) + (lambda_multi * loss_mul)
            valid24_loss += float(loss24.item())

            pred2 = out2.argmax(dim=1)
            pred4 = out4.argmax(dim=1)
            valid2_correct += (pred2 == y_bin).sum().item()
            valid4_correct += (pred4 == y_multi).sum().item()

            for i in (0, 1):
                tp2[i] += ((pred2 == i) & (y_bin == i)).sum().item()
                fp2[i] += ((pred2 == i) & (y_bin != i)).sum().item()
                tn2[i] += ((pred2 != i) & (y_bin != i)).sum().item()
                fn2[i] += ((pred2 != i) & (y_bin == i)).sum().item()

            for i in (0, 1, 2, 3):
                tp4[i] += ((pred4 == i) & (y_multi == i)).sum().item()
                fp4[i] += ((pred4 == i) & (y_multi != i)).sum().item()
                tn4[i] += ((pred4 != i) & (y_multi != i)).sum().item()
                fn4[i] += ((pred4 != i) & (y_multi == i)).sum().item()

    n_batches = len(dataloader)
    valid_loss = valid24_loss / max(n_batches, 1)
    acc2 = 100.0 * valid2_correct / max(n_samples, 1)
    acc4 = 100.0 * valid4_correct / max(n_samples, 1)

    def per_class_metrics(tp, fp, tn, fn):
        prec = [t / (t + f + EPS) for t, f in zip(tp, fp)]
        rec  = [t / (t + f + EPS) for t, f in zip(tp, fn)]
        spec = [t / (t + f + EPS) for t, f in zip(tn, fp)]
        f1   = [2*p*r / (p + r + EPS) for p, r in zip(prec, rec)]
        return prec, rec, spec, f1, {
            'precision_macro': sum(prec)/len(prec),
            'recall_macro': sum(rec)/len(rec),
            'specificity_macro': sum(spec)/len(spec),
            'f1_macro': sum(f1)/len(f1),
        }

    prec2, rec2, spec2, f12, macro2 = per_class_metrics(tp2, fp2, tn2, fn2)
    metrics2 = {'acc': acc2, 'precision': prec2, 'recall': rec2, 'specificity': spec2, 'f1-score': f12, **macro2}

    prec4, rec4, spec4, f14, macro4 = per_class_metrics(tp4, fp4, tn4, fn4)
    metrics4 = {'acc': acc4, 'precision': prec4, 'recall': rec4, 'specificity': spec4, 'f1-score': f14, **macro4}

    return valid_loss, metrics2, metrics4

In [9]:
import gc
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.model_selection import StratifiedKFold
from itertools import chain

from imblearn.over_sampling import SMOTE

EPS = 1e-8


def _clear_cuda_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.no_grad()
@torch.no_grad()
def _brevitas_warmup_rr_both(model24, dataset, device, num_steps=10, bs=8, n_steps=10):
    """Brevitas scaling warmup for RR-to-both model"""
    loader = DataLoader(dataset, batch_size=bs, shuffle=True, num_workers=0,
                        pin_memory=(device.type == "cuda"))
    it = iter(loader)
    model24["net24"].train()
    model24["net2"].train()
    model24["net4"].train()

    for _ in range(n_steps):
        try:
            x, _ = next(it)
        except StopIteration:
            it = iter(loader)
            x, _ = next(it)
        x = x.to(device, non_blocking=True)
        assert x.size(-1) == 184, f"Expected x[...,184] but got {tuple(x.shape)}"
        spk2, _, spk4, _ = forward_pass_rr_both(model24, num_steps, x)
        _ = spk2.mean(0)
        _ = spk4.mean(0)



class TwoHeadCE:
    """
    Single callable loss compatible with your existing train_epoch() / validation_epoch().
    - If logits has C=2 -> uses ce2 (binary)
    - If logits has C=4 -> uses ce4 (multi)
    """
    def __init__(self, ce2, ce4):
        self.ce2 = ce2
        self.ce4 = ce4

    def __call__(self, logits, targets):
        c = logits.size(1)
        if c == 2:
            return self.ce2(logits, targets)
        if c == 4:
            return self.ce4(logits, targets)
        raise ValueError(f"Unexpected logits shape {tuple(logits.shape)}; expected C=2 or C=4.")



In [10]:
def train_model24_cv_smote_rr_both(
    model_factory,
    epochs,
    dataset,
    device,
    loss_func,
    optimizer_class,
    optimizer_kwargs,
    num_steps=10,
    k_folds=6,
    batch_size=128,
    mode="max",
    seed=42,
    patience=10,
    smote_k_neighbors=5,
    monitor_bin="f2_abnormal",
    monitor_multi="f1_macro",
    beta_abn=2.0,
    w_bin=0.7,
    w_multi=0.3,
):
    """
    K-fold CV for dual-head dict-model with SMOTE — RR features routed to BOTH stages.
    """

    if isinstance(device, str):
        device = torch.device(device)

    def all_labels(ds):
        return ds.tensors[1].detach().cpu().long().numpy()

    def move_to_device(m):
        m["net24"].to(device)
        m["net2"].to(device)
        m["net4"].to(device)
        return m

    @torch.no_grad()
    def get_state_cpu(m):
        return {
            "net24": {k: v.detach().cpu().clone() for k, v in m["net24"].state_dict().items()},
            "net2":  {k: v.detach().cpu().clone() for k, v in m["net2"].state_dict().items()},
            "net4":  {k: v.detach().cpu().clone() for k, v in m["net4"].state_dict().items()},
        }

    def load_state_cpu(m, state_cpu):
        m["net24"].load_state_dict(state_cpu["net24"], strict=True)
        m["net2"].load_state_dict(state_cpu["net2"], strict=True)
        m["net4"].load_state_dict(state_cpu["net4"], strict=True)

    def scaling_key_count(state_cpu):
        cnt = 0
        for part in ("net24", "net2", "net4"):
            cnt += sum("scaling_impl.value" in k for k in state_cpu[part].keys())
        return cnt

    def subset_to_xy(sub: Subset):
        base = sub.dataset
        idx = torch.as_tensor(sub.indices, dtype=torch.long)
        X = base.tensors[0][idx]
        y = base.tensors[1][idx]
        return X, y

    def make_smote_dataset(train_subset: Subset):
        X, y = subset_to_xy(train_subset)
        Xn = X.detach().cpu().numpy()
        yn = y.detach().cpu().long().numpy()

        if Xn.ndim != 3 or Xn.shape[1] != 1 or Xn.shape[2] != 184:
            raise ValueError(f"Expected X shape (N,1,184), got {Xn.shape}")

        X_flat = Xn[:, 0, :]

        counts = np.bincount(yn, minlength=4)
        min_class = int(counts[counts > 0].min()) if np.any(counts > 0) else 0
        if min_class < 2:
            print(f"WARNING: min class count={min_class} < 2; skipping SMOTE for this fold.")
            return TensorDataset(X.detach().cpu(), y.detach().cpu())

        k = min(smote_k_neighbors, min_class - 1)
        if k < 1:
            print(f"WARNING: computed k_neighbors={k} < 1; skipping SMOTE for this fold.")
            return TensorDataset(X.detach().cpu(), y.detach().cpu())

        sm = SMOTE(random_state=seed, k_neighbors=k)
        X_res, y_res = sm.fit_resample(X_flat, yn)

        X_res = X_res[:, None, :]

        X_t = torch.from_numpy(X_res).to(dtype=X.dtype)
        y_t = torch.from_numpy(y_res).long()
        return TensorDataset(X_t, y_t)

    def _fbeta(p, r, beta=2.0, eps=1e-8):
        b2 = beta * beta
        return (1 + b2) * p * r / (b2 * p + r + eps)

    def _get_bin_score(v_metrics2, monitor_bin="f2_abnormal", beta_abn=2.0):
        if monitor_bin in ("acc", "precision_macro", "recall_macro", "specificity_macro", "f1_macro"):
            return float(v_metrics2[monitor_bin])

        pA = float(v_metrics2["precision"][1])
        rA = float(v_metrics2["recall"][1])
        f1A = float(v_metrics2["f1-score"][1])

        if monitor_bin == "precision_abnormal":
            return pA
        if monitor_bin == "recall_abnormal":
            return rA
        if monitor_bin == "f1_abnormal":
            return f1A
        if monitor_bin == "f2_abnormal":
            return float(_fbeta(pA, rA, beta=beta_abn, eps=EPS))

        raise ValueError(
            f"Unknown monitor_bin='{monitor_bin}'. "
            f"Use: acc, f1_macro, precision_macro, recall_macro, specificity_macro, "
            f"precision_abnormal, recall_abnormal, f1_abnormal, f2_abnormal."
        )

    def _get_multi_score(v_metrics4, monitor_multi="f1_macro"):
        if monitor_multi in ("acc", "precision_macro", "recall_macro", "specificity_macro", "f1_macro"):
            return float(v_metrics4[monitor_multi])
        raise ValueError(
            f"Unknown monitor_multi='{monitor_multi}'. "
            f"Use: acc, f1_macro, precision_macro, recall_macro, specificity_macro."
        )

    better = (lambda a, b: a > b) if mode == "max" else (lambda a, b: a < b)

    # Brevitas warmup ONCE
    torch.manual_seed(seed)
    np.random.seed(seed)

    print("Creating template model for warmup (RR to both stages)...")
    model_template = model_factory()
    model_template = move_to_device(model_template)

    print("Running Brevitas warmup (RR to both stages)...")
    _brevitas_warmup_rr_both(model_template, dataset, device, num_steps=num_steps, bs=8, n_steps=10)

    warmup_state_cpu = get_state_cpu(model_template)
    print(f"Scaling keys after warmup: {scaling_key_count(warmup_state_cpu)}")

    del model_template
    _clear_cuda_cache()

    # CV split
    y_all = all_labels(dataset)
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)

    fold_history = {}
    fold_ckpts = {}

    for fold, (train_idx, valid_idx) in enumerate(skf.split(np.arange(len(y_all)), y_all), 1):
        print(f"\n{'='*60}\nFold {fold}/{k_folds} (RR to both stages)\n{'='*60}")

        model24_fold = model_factory()
        load_state_cpu(model24_fold, warmup_state_cpu)
        model24_fold = move_to_device(model24_fold)

        params = chain(
            model24_fold["net24"].parameters(),
            model24_fold["net2"].parameters(),
            model24_fold["net4"].parameters(),
        )
        optimizer_fold = optimizer_class(params, **optimizer_kwargs)

        train_subset = Subset(dataset, train_idx)
        valid_subset = Subset(dataset, valid_idx)

        train_smote_ds = make_smote_dataset(train_subset)

        train_loader = DataLoader(
            train_smote_ds,
            batch_size=batch_size,
            shuffle=True,
            pin_memory=(device.type == "cuda"),
            num_workers=0,
        )
        valid_loader = DataLoader(
            valid_subset,
            batch_size=batch_size,
            shuffle=False,
            pin_memory=(device.type == "cuda"),
            num_workers=0,
        )

        y_train = train_smote_ds.tensors[1].detach().cpu().long().numpy()

        n0 = int((y_train == 0).sum())
        n1 = int((y_train == 1).sum())
        n2 = int((y_train == 2).sum())
        n3 = int((y_train == 3).sum())
        total = len(y_train)

        print(f"SMOTE-train counts: Normal={n0}, SVEB={n1}, VEB={n2}, F={n3}")

        n_norm = n0
        n_abn = n1 + n2 + n3
        wN = total / (2 * max(n_norm, 1))
        wA = total / (2 * max(n_abn, 1))
        print(f"Binary counts: Normal={n_norm}, Abnormal={n_abn}")
        print(f"Binary weights: wN={wN:.3f}, wA={wA:.3f}")
        class_w2 = torch.tensor([wN, wA], dtype=torch.float32, device=device)
        ce2 = torch.nn.CrossEntropyLoss(weight=class_w2)

        ce4 = torch.nn.CrossEntropyLoss()

        loss_fold = TwoHeadCE(ce2=ce2, ce4=ce4)

        epoch_history = {
            "train_loss": [], "valid_loss": [],
            "train2_acc": [], "train4_acc": [],
            "valid2_acc": [], "valid2_prec": [], "valid2_rec": [], "valid2_spec": [], "valid2_f1": [],
            "valid2_prec_macro": [], "valid2_rec_macro": [], "valid2_spec_macro": [], "valid2_f1_macro": [],
            "valid4_acc": [], "valid4_prec": [], "valid4_rec": [], "valid4_spec": [], "valid4_f1": [],
            "valid4_prec_macro": [], "valid4_rec_macro": [], "valid4_spec_macro": [], "valid4_f1_macro": [],
        }

        best_score = -float("inf") if mode == "max" else float("inf")
        best_epoch = None
        best_state_cpu = None
        best_metrics2 = None
        best_metrics4 = None
        best_bin_score = None
        best_multi_score = None
        no_improve = 0

        n_train_samples = len(train_smote_ds)
        n_train_batches = len(train_loader)

        for epoch in range(1, epochs + 1):
            # Use RR-both versions
            tr_loss_sum, tr2_correct, tr4_correct = train_epoch_rr_both(
                model24_fold, train_loader, loss_fold, optimizer_fold, device, num_steps
            )

            train_loss = tr_loss_sum / max(n_train_batches, 1)
            tr2_acc = 100.0 * tr2_correct / max(n_train_samples, 1)
            tr4_acc = 100.0 * tr4_correct / max(n_train_samples, 1)

            valid_loss, v_metrics2, v_metrics4 = validation_epoch_rr_both(
                model24_fold, valid_loader, loss_fold, device, num_steps
            )

            pA = float(v_metrics2["precision"][1])
            rA = float(v_metrics2["recall"][1])
            f2A = float(_fbeta(pA, rA, beta=beta_abn, eps=EPS))

            print(
                f"[Fold {fold}][Epoch {epoch:2d}/{epochs}] "
                f"Train L={train_loss:.4f} | "
                f"Val L={valid_loss:.4f} | "
                f"BinAcc={v_metrics2['acc']:5.2f}% "
                f"(P_A={pA:.4f} R_A={rA:.4f} F2_A={f2A:.4f}) | "
                f"MultiAcc={v_metrics4['acc']:5.2f}% F1m4={v_metrics4['f1_macro']:.4f}"
            )

            epoch_history["train_loss"].append(train_loss)
            epoch_history["valid_loss"].append(valid_loss)
            epoch_history["train2_acc"].append(tr2_acc)
            epoch_history["train4_acc"].append(tr4_acc)

            epoch_history["valid2_acc"].append(v_metrics2["acc"])
            epoch_history["valid2_prec"].append(v_metrics2["precision"])
            epoch_history["valid2_rec"].append(v_metrics2["recall"])
            epoch_history["valid2_spec"].append(v_metrics2["specificity"])
            epoch_history["valid2_f1"].append(v_metrics2["f1-score"])
            epoch_history["valid2_prec_macro"].append(v_metrics2["precision_macro"])
            epoch_history["valid2_rec_macro"].append(v_metrics2["recall_macro"])
            epoch_history["valid2_spec_macro"].append(v_metrics2["specificity_macro"])
            epoch_history["valid2_f1_macro"].append(v_metrics2["f1_macro"])

            epoch_history["valid4_acc"].append(v_metrics4["acc"])
            epoch_history["valid4_prec"].append(v_metrics4["precision"])
            epoch_history["valid4_rec"].append(v_metrics4["recall"])
            epoch_history["valid4_spec"].append(v_metrics4["specificity"])
            epoch_history["valid4_f1"].append(v_metrics4["f1-score"])
            epoch_history["valid4_prec_macro"].append(v_metrics4["precision_macro"])
            epoch_history["valid4_rec_macro"].append(v_metrics4["recall_macro"])
            epoch_history["valid4_spec_macro"].append(v_metrics4["specificity_macro"])
            epoch_history["valid4_f1_macro"].append(v_metrics4["f1_macro"])

            bin_score = _get_bin_score(v_metrics2, monitor_bin=monitor_bin, beta_abn=beta_abn)
            multi_score = _get_multi_score(v_metrics4, monitor_multi=monitor_multi)
            score = (w_bin * bin_score) + (w_multi * multi_score)

            if better(score, best_score):
                best_score = score
                best_epoch = epoch
                no_improve = 0
                best_state_cpu = get_state_cpu(model24_fold)
                best_bin_score = float(bin_score)
                best_multi_score = float(multi_score)

                best_metrics2 = {
                    "acc": v_metrics2["acc"],
                    "precision_macro": v_metrics2["precision_macro"],
                    "recall_macro": v_metrics2["recall_macro"],
                    "specificity_macro": v_metrics2["specificity_macro"],
                    "f1_macro": v_metrics2["f1_macro"],
                    "precision_abnormal": float(v_metrics2["precision"][1]),
                    "recall_abnormal": float(v_metrics2["recall"][1]),
                    "f1_abnormal": float(v_metrics2["f1-score"][1]),
                    "f2_abnormal": float(_fbeta(float(v_metrics2["precision"][1]),
                                                float(v_metrics2["recall"][1]),
                                                beta=beta_abn, eps=EPS)),
                }
                best_metrics4 = {
                    "acc": v_metrics4["acc"],
                    "precision_macro": v_metrics4["precision_macro"],
                    "recall_macro": v_metrics4["recall_macro"],
                    "specificity_macro": v_metrics4["specificity_macro"],
                    "f1_macro": v_metrics4["f1_macro"],
                }
            else:
                if epoch >= 20:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f"Early stopping: no improvement in {patience} epochs.")
                        break

        fold_key = f"fold{fold}"
        fold_history[fold_key] = epoch_history
        fold_ckpts[fold_key] = {
            "best_epoch": best_epoch,
            "best_score": float(best_score),
            "mode": mode,
            "monitor_bin": monitor_bin,
            "monitor_multi": monitor_multi,
            "beta_abn": float(beta_abn),
            "w_bin": float(w_bin),
            "w_multi": float(w_multi),
            "best_bin_score": float(best_bin_score) if best_bin_score is not None else None,
            "best_multi_score": float(best_multi_score) if best_multi_score is not None else None,
            "state_cpu": best_state_cpu,
            "best_metrics2": best_metrics2,
            "best_metrics4": best_metrics4,
        }

        print(
            f"\n[Fold {fold}] Best combined score={best_score:.6f} "
            f"(bin={best_bin_score:.6f}, multi={best_multi_score:.6f}) at epoch {best_epoch}\n"
        )

        del optimizer_fold, model24_fold, train_loader, valid_loader, train_smote_ds
        _clear_cuda_cache()

    return fold_history, fold_ckpts

In [13]:
 model_factory=create_qcsnn_model24_rr_both()

Output sizes: Block1=89, Block2=43, Block3=20
Flattened size: 480
Stage 1 input: 484 (trunk + RR)
Stage 2 input: 484 (trunk + RR)


In [15]:
model = create_qcsnn_model24_rr_both()
total = 0
for key, subnet in model.items():
    subnet_params = sum(p.numel() for p in subnet.parameters())
    print(f"{key}: {subnet_params}")
    total += subnet_params
print(f"\nTotal parameters: {total}")

Output sizes: Block1=89, Block2=43, Block3=20
Flattened size: 480
Stage 1 input: 484 (trunk + RR)
Stage 2 input: 484 (trunk + RR)
net24: 2095
net2: 972
net4: 62472

Total parameters: 65539


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# Step 1: Load Data
train_data, train_targets, rr_stats = load_train_test_data(return_rr_stats=True)

# Step 2: Create Dataset
dataset = create_csnn_datasets(train_data, train_targets)

# Step 3: Run CV with SMOTE — RR to Both Stages
fold_history_rr_both, fold_ckpts_rr_both = train_model24_cv_smote_rr_both(
    model_factory=create_qcsnn_model24_rr_both,
    epochs=80,
    dataset=dataset,
    device=device,
    loss_func=None,
    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={'lr': 0.0001},
    num_steps=10,
    k_folds=6,
    batch_size=128,
    mode="max",
    seed=42,
    patience=10,
    smote_k_neighbors=5,
    monitor_bin="f2_abnormal",
    monitor_multi="f1_macro",
    beta_abn=2.0,
    w_bin=0.7,
    w_multi=0.3,
)

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from snntorch import utils

EPS = 1e-8


# -------------------------
# Helpers
# -------------------------
def _split_signal_and_rr(x, signal_len=180, rr_dim=4):
    """
    Supports x shaped:
      - (B, signal_len + rr_dim)
      - (B, 1, signal_len + rr_dim)
      - (B, 1, signal_len)   -> rr=None
      - (B, signal_len)      -> rr=None
    Returns:
      x_sig: same rank as input (keeps channel dim if present)
      rr:    (B, rr_dim) or None
    """
    if rr_dim is None or rr_dim <= 0:
        return x, None

    if x.dim() == 2:
        # (B, Ltot)
        if x.size(1) < signal_len:
            raise RuntimeError(f"x has {x.size(1)} features, expected at least signal_len={signal_len}.")
        x_sig = x[:, :signal_len]
        rr = None
        if x.size(1) >= signal_len + rr_dim:
            rr = x[:, signal_len:signal_len + rr_dim]
        return x_sig, rr

    if x.dim() == 3:
        # (B, C, Ltot)
        if x.size(2) < signal_len:
            raise RuntimeError(f"x has length {x.size(2)}, expected at least signal_len={signal_len}.")
        x_sig = x[:, :, :signal_len]
        rr = None
        if x.size(2) >= signal_len + rr_dim:
            rr_raw = x[:, :, signal_len:signal_len + rr_dim]  # (B, C, rr_dim)
            rr = rr_raw[:, 0, :] if rr_raw.size(1) > 1 else rr_raw.squeeze(1)  # (B, rr_dim)
        return x_sig, rr

    raise RuntimeError(f"Unsupported x.dim()={x.dim()} (expected 2 or 3).")


def _infer_first_linear_in_features(module):
    """
    Finds the first nn.Linear-like module with an 'in_features' attribute.
    Works with Brevitas QuantLinear too (it exposes in_features).
    """
    for m in module.modules():
        if hasattr(m, "in_features"):
            return int(m.in_features)
    return None


# -------------------------
# Scoring helpers
# -------------------------
def _fbeta(p, r, beta=2.0, eps=EPS):
    b2 = beta * beta
    return (1 + b2) * p * r / (b2 * p + r + eps)


def _get_bin_score(v_metrics2, monitor_bin="f2_abnormal", beta_abn=2.0):
    if monitor_bin in ("acc", "precision_macro", "recall_macro", "specificity_macro", "f1_macro"):
        return float(v_metrics2[monitor_bin])

    pA = float(v_metrics2["precision"][1])
    rA = float(v_metrics2["recall"][1])
    f1A = float(v_metrics2["f1-score"][1])

    if monitor_bin == "precision_abnormal":
        return pA
    if monitor_bin == "recall_abnormal":
        return rA
    if monitor_bin == "f1_abnormal":
        return f1A
    if monitor_bin == "f2_abnormal":
        return float(_fbeta(pA, rA, beta=beta_abn, eps=EPS))

    raise ValueError(
        f"Unknown monitor_bin='{monitor_bin}'. "
        f"Use: acc, f1_macro, precision_macro, recall_macro, specificity_macro, "
        f"precision_abnormal, recall_abnormal, f1_abnormal, f2_abnormal."
    )


def _get_multi_score(v_metrics4, monitor_multi="f1_macro"):
    if monitor_multi not in v_metrics4:
        raise ValueError(f"Unknown monitor_multi='{monitor_multi}'. Available keys: {list(v_metrics4.keys())}")
    return float(v_metrics4[monitor_multi])


def _per_class_metrics(tp, fp, tn, fn):
    prec = [t / (t + f + EPS) for t, f in zip(tp, fp)]
    rec  = [t / (t + f + EPS) for t, f in zip(tp, fn)]
    spec = [t / (t + f + EPS) for t, f in zip(tn, fp)]
    f1   = [2*p*r / (p + r + EPS) for p, r in zip(prec, rec)]
    return {
        "precision": prec,
        "recall": rec,
        "specificity": spec,
        "f1-score": f1,
        "precision_macro": sum(prec)/len(prec),
        "recall_macro": sum(rec)/len(rec),
        "specificity_macro": sum(spec)/len(spec),
        "f1_macro": sum(f1)/len(f1),
    }


# -------------------------
# Two-stage forward (Stage 2 can predict any class)
# -------------------------
@torch.no_grad()
def forward_two_stage_rr_both(net, num_steps, x, gate="argmax", signal_len=180, rr_dim=4):
    """Two-stage inference with RR routed to BOTH stages"""
    net24, net2, net4 = net["net24"], net["net2"], net["net4"]
    B = x.size(0)
    device = x.device

    utils.reset(net24)
    utils.reset(net2)
    utils.reset(net4)

    x_sig, rr = _split_signal_and_rr(x, signal_len=signal_len, rr_dim=rr_dim)
    x_sig = x_sig.to(device)
    if rr is not None:
        rr = rr.to(device)

    trunk_cache_aug = []
    spk2_rec = []

    for _ in range(num_steps):
        trunk = net24(x_sig)
        aug = torch.cat([trunk, rr], dim=1) if rr is not None else trunk
        trunk_cache_aug.append(aug)
        spk2, _ = net2(aug)
        spk2_rec.append(spk2)

    logits2 = torch.stack(spk2_rec, dim=0).mean(0)
    pred2 = logits2.argmax(dim=1)
    routed_mask = (pred2 == 1)

    final_pred4 = torch.zeros(B, dtype=torch.long, device=device)

    if routed_mask.any():
        idx = routed_mask.nonzero(as_tuple=False).squeeze(1)
        utils.reset(net4)

        spk4_rec = []
        for t in range(num_steps):
            out_t = trunk_cache_aug[t].index_select(0, idx)
            # NO stripping — Stage 2 gets full 484
            spk4, _ = net4(out_t)
            spk4_rec.append(spk4)

        logits4 = torch.stack(spk4_rec, dim=0).mean(0)
        pred4 = logits4.argmax(dim=1)
        final_pred4.index_copy_(0, idx, pred4)

    return pred2, final_pred4, routed_mask

# -------------------------
# Evaluator-old
# -------------------------
@torch.no_grad()
def evaluate_loader_two_stage_rr_both(model24, loader, device, num_steps=10, signal_len=180, rr_dim=4):
    """Evaluation using RR to both stages"""
    model24["net24"].eval()
    model24["net2"].eval()
    model24["net4"].eval()

    tp2 = [0, 0]; fp2 = [0, 0]; tn2 = [0, 0]; fn2 = [0, 0]
    tp4 = [0, 0, 0, 0]; fp4 = [0, 0, 0, 0]; tn4 = [0, 0, 0, 0]; fn4 = [0, 0, 0, 0]

    n, correct_final, routed_total, correct_stage1 = 0, 0, 0, 0

    for xb, y4 in loader:
        xb = xb.to(device, non_blocking=True)
        y4 = y4.to(device, non_blocking=True).long()
        y2 = (y4 > 0).long()
        B = xb.size(0)
        n += B

        pred2, final4, routed_mask = forward_two_stage_rr_both(
            model24, num_steps, xb, gate="argmax", signal_len=signal_len, rr_dim=rr_dim
        )

        routed_total += int(routed_mask.sum().item())
        correct_stage1 += (pred2 == y2).sum().item()
        correct_final += (final4 == y4).sum().item()

        for c in (0, 1):
            tp2[c] += ((pred2 == c) & (y2 == c)).sum().item()
            fp2[c] += ((pred2 == c) & (y2 != c)).sum().item()
            tn2[c] += ((pred2 != c) & (y2 != c)).sum().item()
            fn2[c] += ((pred2 != c) & (y2 == c)).sum().item()

        for c in (0, 1, 2, 3):
            tp4[c] += ((final4 == c) & (y4 == c)).sum().item()
            fp4[c] += ((final4 == c) & (y4 != c)).sum().item()
            tn4[c] += ((final4 != c) & (y4 != c)).sum().item()
            fn4[c] += ((final4 != c) & (y4 == c)).sum().item()

    stage1_acc = 100.0 * correct_stage1 / max(n, 1)
    final_acc = 100.0 * correct_final / max(n, 1)
    routed_pct = 100.0 * routed_total / max(n, 1)

    m2 = {"acc": stage1_acc, **_per_class_metrics(tp2, fp2, tn2, fn2)}
    m4 = {"acc": final_acc, **_per_class_metrics(tp4, fp4, tn4, fn4)}
    return m2, m4, routed_pct


# -------------------------
# Fold evaluator
# -------------------------
def _all_labels(ds):
    if hasattr(ds, "tensors") and len(ds.tensors) >= 2:
        return ds.tensors[1].detach().cpu().long().numpy()
    return np.asarray([ds[i][1] for i in range(len(ds))], dtype=np.int64)

def _load_state_cpu(model24, state_cpu):
    model24["net24"].load_state_dict(state_cpu["net24"], strict=True)
    model24["net2"].load_state_dict(state_cpu["net2"], strict=True)
    model24["net4"].load_state_dict(state_cpu["net4"], strict=True)

def _move_to_device(model24, device):
    model24["net24"].to(device)
    model24["net2"].to(device)
    model24["net4"].to(device)
    return model24


def evaluate_all_folds_and_pick_best_rr_both(
    fold_ckpts,
    model_factory,
    dataset,
    device,
    num_steps=10,
    k_folds=6,
    batch_size=128,
    seed=42,

    best_by="combined",
    monitor_bin="f2_abnormal",
    beta_abn=2.0,
    monitor_multi="f1_macro",
    w_bin=0.7,
    w_multi=0.3,

    signal_len=180,
    rr_dim=4,
):
    y_all = _all_labels(dataset)
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)

    results = {}

    for fold, (_, valid_idx) in enumerate(skf.split(np.arange(len(y_all)), y_all), 1):
        fold_key = f"fold{fold}"
        state_cpu = fold_ckpts[fold_key]["state_cpu"]

        model24 = model_factory()
        _load_state_cpu(model24, state_cpu)
        model24 = _move_to_device(model24, device)

        valid_loader = DataLoader(
            Subset(dataset, valid_idx),
            batch_size=batch_size,
            shuffle=False,
            pin_memory=(device.type == "cuda"),
            num_workers=0,
        )

        # CHANGED: Use RR-to-both evaluation
        m2, m4, routed_pct = evaluate_loader_two_stage_rr_both(
            model24, valid_loader, device,
            num_steps=num_steps,
            signal_len=signal_len,
            rr_dim=rr_dim
        )

        bin_score = _get_bin_score(m2, monitor_bin=monitor_bin, beta_abn=beta_abn)
        multi_score = _get_multi_score(m4, monitor_multi=monitor_multi)
        combined = w_bin * bin_score + w_multi * multi_score

        results[fold_key] = {
            "routed_%": float(routed_pct),
            "stage1_binary": m2,
            "final_two_stage": m4,
            "bin_score": float(bin_score),
            "multi_score": float(multi_score),
            "combined_score": float(combined),
        }

        print(
            f"[{fold_key}] routed={routed_pct:5.2f}% | "
            f"Stage1: Acc={m2['acc']:5.2f}% {monitor_bin}={bin_score:.4f} | "
            f"Final:  Acc={m4['acc']:5.2f}% {monitor_multi}={multi_score:.4f} | "
            f"Combined={combined:.6f}"
        )

        del model24
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # Pick best fold
    if best_by == "combined":
        key_fn = lambda fk: results[fk]["combined_score"]
    elif best_by == "final_f1_macro":
        key_fn = lambda fk: results[fk]["final_two_stage"]["f1_macro"]
    elif best_by == "final_acc":
        key_fn = lambda fk: results[fk]["final_two_stage"]["acc"]
    elif best_by == "stage1_score":
        key_fn = lambda fk: results[fk]["bin_score"]
    else:
        raise ValueError("best_by must be one of: combined, final_f1_macro, final_acc, stage1_score")

    best_fold = max(results.keys(), key=key_fn)
    best_score = key_fn(best_fold)

    print("\n==============================")
    print(f"✅ BEST FOLD by {best_by}: {best_fold}  (score={best_score:.6f})")
    print("==============================\n")

    return best_fold, results

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_fold, fold_eval_results = evaluate_all_folds_and_pick_best_rr_both(
    fold_ckpts=fold_ckpts_rr_both,
    model_factory=create_qcsnn_model24_rr_both,
    dataset=dataset,
    device=device,
    num_steps=10,
    k_folds=6,
    batch_size=128,
    seed=42,

    best_by="combined",
    monitor_bin="f2_abnormal",
    beta_abn=2.0,
    monitor_multi="f1_macro",
    w_bin=0.7,
    w_multi=0.3,

    signal_len=180,
    rr_dim=4,
)


In [ ]:
import numpy as np
import torch
import torch.nn as nn


class TwoHeadCE:
    """
    Single callable loss compatible with your existing train_epoch() / validation_epoch().
    - If logits has C=2 -> uses ce2 (binary)
    - If logits has C=4 -> uses ce4 (multi)
    """
    def __init__(self, ce2, ce4):
        self.ce2 = ce2
        self.ce4 = ce4

    def __call__(self, logits, targets):
        c = logits.size(1)
        if c == 2:
            return self.ce2(logits, targets)
        if c == 4:
            return self.ce4(logits, targets)
        raise ValueError(f"Unexpected logits shape {tuple(logits.shape)}; expected C=2 or C=4.")

def build_fulltrain_weighted_ce(train_dataset, device):
    """
    Same weighting rule as CV SMOTE training:
    - Binary head: weighted CE
    - Multi head: unweighted CE (SMOTE already balanced)
    """
    if hasattr(train_dataset, "tensors") and len(train_dataset.tensors) >= 2:
        y = train_dataset.tensors[1].detach().cpu().long().numpy()
    else:
        y = np.asarray([train_dataset[i][1] for i in range(len(train_dataset))], dtype=np.int64)

    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    n2 = int((y == 2).sum())
    n3 = int((y == 3).sum())
    total = len(y)

    print(f"FULL TRAIN counts: N={n0}, SVEB={n1}, VEB={n2}, F={n3} (total={total})")

    # Binary weights: Normal vs Abnormal
    n_norm = n0
    n_abn = n1 + n2 + n3
    wN = total / (2 * max(n_norm, 1))
    wA = total / (2 * max(n_abn, 1))
    class_w2 = torch.tensor([wN, wA], dtype=torch.float32, device=device)
    ce2 = torch.nn.CrossEntropyLoss(weight=class_w2)
    print(f"Binary weights: wN={wN:.3f}, wA={wA:.3f}")

    # Multi head: unweighted (SMOTE balances classes)
    ce4 = torch.nn.CrossEntropyLoss()
    print("4-class weights: None (unweighted)")

    return TwoHeadCE(ce2=ce2, ce4=ce4)



In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from imblearn.over_sampling import SMOTE
from itertools import chain
import numpy as np

def load_fold_state_into_model(model, state_cpu):
    model["net24"].load_state_dict(state_cpu["net24"], strict=True)
    model["net2"].load_state_dict(state_cpu["net2"], strict=True)
    model["net4"].load_state_dict(state_cpu["net4"], strict=True)
    return model

def move_model_to_device(model, device):
    model["net24"].to(device)
    model["net2"].to(device)
    model["net4"].to(device)
    return model

def finetune_best_fold_on_full_train(
    fold_ckpts,
    model_factory,
    train_dataset,
    device,
    optimizer_class,
    optimizer_kwargs,
    num_steps=10,
    batch_size=128,
    finetune_epochs=5,
    fold_key="fold3",
    lr_scale=0.1,
    use_smote=True,
    smote_k_neighbors=5,
    seed=42,
):
    # --- Load fold checkpoint ---
    state_cpu = fold_ckpts[fold_key]["state_cpu"]
    model = model_factory()
    model = load_fold_state_into_model(model, state_cpu)
    model = move_model_to_device(model, device)

    # --- Apply SMOTE if requested ---
    if use_smote:
        X = train_dataset.tensors[0].detach().cpu().numpy()  # (N, 1, 184)
        y = train_dataset.tensors[1].detach().cpu().long().numpy()

        if X.ndim != 3 or X.shape[1] != 1 or X.shape[2] != 184:
            raise ValueError(f"Expected X shape (N,1,184), got {X.shape}")

        # flatten (N,1,184) -> (N,184)
        X_flat = X[:, 0, :]

        # choose safe k_neighbors
        counts = np.bincount(y, minlength=4)
        min_class = int(counts[counts > 0].min()) if np.any(counts > 0) else 0
        k = min(smote_k_neighbors, min_class - 1) if min_class >= 2 else 0

        if k >= 1:
            sm = SMOTE(random_state=seed, k_neighbors=k)
            X_res, y_res = sm.fit_resample(X_flat, y)
            # reshape back (N',184) -> (N',1,184)
            X_res = X_res[:, None, :]
            X_t = torch.from_numpy(X_res).to(dtype=train_dataset.tensors[0].dtype)
            y_t = torch.from_numpy(y_res).long()
            train_ds = TensorDataset(X_t, y_t)
            print(f"SMOTE applied: {len(y)} -> {len(y_res)} samples")
        else:
            print(f"WARNING: k_neighbors={k} < 1; skipping SMOTE.")
            train_ds = train_dataset
    else:
        train_ds = train_dataset

    # --- Full train loader ---
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device.type == "cuda"),
        num_workers=0,
    )

    # --- Class-weighted CE (computed from SMOTE-balanced data if applied) ---
    loss_fn = build_fulltrain_weighted_ce(train_ds, device)

    # --- Optimizer ---
    opt_kwargs = dict(optimizer_kwargs)
    if "lr" in opt_kwargs:
        opt_kwargs["lr"] = opt_kwargs["lr"] * lr_scale
    params = chain(model["net24"].parameters(), model["net2"].parameters(), model["net4"].parameters())
    optimizer = optimizer_class(params, **opt_kwargs)

    # --- Train ---
    history = {"train_loss": [], "train2_acc": [], "train4_acc": []}
    n_samples = len(train_ds)

    for ep in range(1, finetune_epochs + 1):
        tr_loss_sum, tr2_correct, tr4_correct = train_epoch_rr_both(
            model, train_loader, loss_fn, optimizer, device, num_steps
        )
        train_loss = tr_loss_sum / max(len(train_loader), 1)
        tr2_acc = 100.0 * tr2_correct / max(n_samples, 1)
        tr4_acc = 100.0 * tr4_correct / max(n_samples, 1)

        history["train_loss"].append(train_loss)
        history["train2_acc"].append(tr2_acc)
        history["train4_acc"].append(tr4_acc)

        print(f"[FINETUNE {fold_key}][Epoch {ep:2d}/{finetune_epochs}] "
              f"L={train_loss:.4f} | BinAcc={tr2_acc:.2f}% | MultiAcc={tr4_acc:.2f}%")

    return model, history

In [ ]:
def _per_class_metrics(tp, fp, tn, fn, class_names=None, eps=1e-8):
    """
    Returns:
      - per-class metrics as LISTS (compatible with your current code)
      - per-class metrics as a NAME->VALUE dict (what you want)
      - macro averages
    """
    C = len(tp)
    if class_names is None:
        class_names = [f"class{i}" for i in range(C)]

    prec = [tp[i] / (tp[i] + fp[i] + eps) for i in range(C)]
    rec  = [tp[i] / (tp[i] + fn[i] + eps) for i in range(C)]
    spec = [tn[i] / (tn[i] + fp[i] + eps) for i in range(C)]
    f1   = [2 * prec[i] * rec[i] / (prec[i] + rec[i] + eps) for i in range(C)]

    # Named per-class dict (this is what you want baked in)
    per_class = {
        class_names[i]: {
            "precision": prec[i],
            "recall": rec[i],
            "f1": f1[i],
            "specificity": spec[i],
            "tp": tp[i], "fp": fp[i], "tn": tn[i], "fn": fn[i],
        }
        for i in range(C)
    }

    return {
        
        "precision": prec,
        "recall": rec,
        "specificity": spec,
        "f1-score": f1,

       
        "per_class": per_class,

        # Macro
        "precision_macro": sum(prec) / C,
        "recall_macro": sum(rec) / C,
        "specificity_macro": sum(spec) / C,
        "f1_macro": sum(f1) / C,
    }



@torch.no_grad()
def evaluate_two_stage_rr_both(
    model, dataloader, device, num_steps=10,
    signal_len=180,
    rr_dim=4,
    class_names2=("Normal(0)", "Abnormal(1)"),
    class_names4=("N(0)", "SVEB(1)", "VEB(2)", "F(3)"),
    verbose=True
):
    """Final evaluation with RR routed to BOTH stages"""
    model["net24"].eval()
    model["net2"].eval()
    model["net4"].eval()

    tp2 = [0, 0]; fp2 = [0, 0]; tn2 = [0, 0]; fn2 = [0, 0]
    tp4 = [0, 0, 0, 0]; fp4 = [0, 0, 0, 0]; tn4 = [0, 0, 0, 0]; fn4 = [0, 0, 0, 0]

    n = 0
    correct_final = 0
    correct_stage1 = 0
    routed_total = 0

    for xb, y4 in dataloader:
        xb = xb.to(device, non_blocking=True)
        y4 = y4.to(device, non_blocking=True).long()
        y2 = (y4 > 0).long()

        B = xb.size(0)
        n += B

        # CHANGED: Use RR-to-both forward
        pred2, final4, routed_mask = forward_two_stage_rr_both(
            model, num_steps, xb,
            gate="argmax",
            signal_len=signal_len,
            rr_dim=rr_dim
        )

        routed_total += int(routed_mask.sum().item())
        correct_stage1 += (pred2 == y2).sum().item()
        correct_final  += (final4 == y4).sum().item()

        for c in (0, 1):
            tp2[c] += ((pred2 == c) & (y2 == c)).sum().item()
            fp2[c] += ((pred2 == c) & (y2 != c)).sum().item()
            tn2[c] += ((pred2 != c) & (y2 != c)).sum().item()
            fn2[c] += ((pred2 != c) & (y2 == c)).sum().item()

        for c in (0, 1, 2, 3):
            tp4[c] += ((final4 == c) & (y4 == c)).sum().item()
            fp4[c] += ((final4 == c) & (y4 != c)).sum().item()
            tn4[c] += ((final4 != c) & (y4 != c)).sum().item()
            fn4[c] += ((final4 != c) & (y4 == c)).sum().item()

    stage1_acc = 100.0 * correct_stage1 / max(n, 1)
    final_acc  = 100.0 * correct_final  / max(n, 1)
    routed_pct = 100.0 * routed_total   / max(n, 1)

    m2 = {"acc": stage1_acc, **_per_class_metrics(tp2, fp2, tn2, fn2, class_names=list(class_names2))}
    m4 = {"acc": final_acc,  **_per_class_metrics(tp4, fp4, tn4, fn4, class_names=list(class_names4))}

    if verbose:
        print("\n==================== TWO-STAGE RR-TO-BOTH EVAL ====================")
        print(f"Routed % (sent to net4): {routed_pct:.2f}%")

        print("\n--- Stage-1 Binary Metrics (per-class) ---")
        print(f"Stage-1 Acc: {m2['acc']:.2f}% | Macro F1: {m2['f1_macro']:.4f}")
        for cname, v in m2["per_class"].items():
            print(f"{cname:>15s} | Prec={v['precision']:.4f}  Rec={v['recall']:.4f}  F1={v['f1']:.4f}")

        print("\n--- Final Two-Stage 4-Class Metrics (per-class) ---")
        print(f"Final Acc : {m4['acc']:.2f}% | Macro F1: {m4['f1_macro']:.4f}")
        for cname, v in m4["per_class"].items():
            print(f"{cname:>15s} | Prec={v['precision']:.4f}  Rec={v['recall']:.4f}  F1={v['f1']:.4f}")

        print("===================================================================\n")

    return m2, m4, routed_pct

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
best_fold = "fold3"
# 1) Fine-tune Fold 3 on FULL training dataset
model_final, finetune_hist = finetune_best_fold_on_full_train(
    fold_ckpts=fold_ckpts_rr_both,
    model_factory=create_qcsnn_model24_rr_both,
    train_dataset=dataset,
    device=device,
    optimizer_class=torch.optim.Adam,
    optimizer_kwargs={'lr': 0.0001},
    num_steps=10,
    batch_size=128,
    finetune_epochs=10,
    fold_key=best_fold,
    lr_scale=0.1,
    use_smote=True,
    smote_k_neighbors=5,
    seed=42,
)


In [ ]:
import torch
from datetime import datetime

def save_qcsnn24_dict_state(model_dict, save_path):
    state_cpu = {
        "net24": {k: v.detach().cpu() for k, v in model_dict["net24"].state_dict().items()},
        "net2":  {k: v.detach().cpu() for k, v in model_dict["net2"].state_dict().items()},
        "net4":  {k: v.detach().cpu() for k, v in model_dict["net4"].state_dict().items()},
    }
    torch.save(state_cpu, save_path)
    print(f"[Saved] Fine-tuned state_dict -> {save_path}")
    return state_cpu

def load_qcsnn24_dict_state(model_factory, state_cpu, device="cpu"):
    model = model_factory()  # {"net24","net2","net4"}
    model["net24"].load_state_dict(state_cpu["net24"], strict=True)
    model["net2"].load_state_dict(state_cpu["net2"], strict=True)
    model["net4"].load_state_dict(state_cpu["net4"], strict=True)

    model["net24"].to(device).eval()
    model["net2"].to(device).eval()
    model["net4"].to(device).eval()
    return model


stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f"fold3_with_stage12_rr_features_SMOTE_fulltrain_finetuned_{stamp}.pth"

final_state_cpu = save_qcsnn24_dict_state(model_final, save_path)


In [ ]:
# Load test data (adjust function name if different)
mitbih_test_path='/home/velox-217533/Projects/fau_projects/research/data/mitbih_processed_intra_patient_4class_180_center90_rr_features_filtered/test'

test_data, test_targets = load_train_test_data(mitbih_path=mitbih_test_path, 
                                               rr_stats=rr_stats, 
                                               return_rr_stats=False)
test_dataset = create_csnn_datasets(test_data, test_targets)

print(f"Test set size: {len(test_data)}")
print(f"Test normal samples: {(test_targets == 0).sum()}")
print(f"Test sveb samples: {(test_targets == 1).sum()}")
print(f"Test veb samples: {(test_targets == 2).sum()}")
print(f"Test f samples: {(test_targets == 3).sum()}")

In [ ]:
# 2) Evaluate on UNSEEN test set using strict cascade Option-A
test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    pin_memory=(device.type == "cuda"),
    num_workers=0,
)

# m2_test, m4_test, routed_test = evaluate_two_stage_optionA(
#     model_final, test_loader, device, num_steps=10
# )
m2_test, m4_test, routed_test = evaluate_two_stage_rr_both(
    model_final, test_loader, device, 
    num_steps=10,
    signal_len=180,
    rr_dim=4
)


# print("\n=== TEST (Two-Stage Option-A) ===")
# print(f"Routed %: {routed_test:.2f}%")
# print(f"Stage1 Binary: Acc={m2_test['acc']:.2f}%  F1_macro={m2_test['f1_macro']:.4f}")
# print(f"Final 4-class: Acc={m4_test['acc']:.2f}%  F1_macro={m4_test['f1_macro']:.4f}")


In [ ]:


import os
import math
import numpy as np
import torch

import brevitas.nn as qnn
import snntorch as snn

# -----------------------
# Utilities
# -----------------------

def _ensure_dir(d):
    os.makedirs(d, exist_ok=True)

def _to_one(x):
    if isinstance(x, torch.Tensor):
        if x.numel() == 1:
            return float(x.detach().cpu().item())
        return x.detach().cpu().numpy()
    if isinstance(x, (float, int)):
        return x
    return x

def _get_bit_scale_zp_from_quant(q):
    """Return (bit_width, scale, zero_point, signed) from a brevitas quantizer-like object."""
    bit_width = None
    scale = None
    zero_point = None
    signed = True
    # bit width
    for k in ('bit_width', 'bit_width_impl', 'bit_width_f'):
        v = getattr(q, k, None)
        if v is None: continue
        try:
            v = v() if callable(v) else v
            bit_width = int(_to_one(v))
            break
        except Exception:
            pass
    # scale
    for k in ('scale', 'tensor_scale', 'scale_impl', 'act_scale', 'weight_scale'):
        v = getattr(q, k, None)
        if v is None: continue
        try:
            v = v() if callable(v) else v
            v = _to_one(v)
            if isinstance(v, (float, int)):
                scale = float(v)
                break
            if isinstance(v, np.ndarray) and v.size == 1:
                scale = float(v.item())
                break
        except Exception:
            pass
    # zero-point
    for k in ('zero_point', 'zero_point_impl', 'zp'):
        v = getattr(q, k, None)
        if v is None: continue
        try:
            v = v() if callable(v) else v
            zero_point = int(round(_to_one(v)))
            break
        except Exception:
            pass
    # signed
    sattr = getattr(q, 'signed', None)
    if isinstance(sattr, bool):
        signed = sattr
    elif zero_point is None:
        signed = True
    # defaults
    if bit_width is None: bit_width = 8
    if scale is None:     scale = 1.0
    if zero_point is None: zero_point = 0
    return bit_width, float(scale), int(zero_point), bool(signed)

def _quantize_multiplier(real_multiplier: float):
    """TFLite-style integer multiplier/shift approximation for a positive real multiplier."""
    if real_multiplier <= 0.0:
        return 0, 0
    mantissa, exponent = math.frexp(real_multiplier)  # real = mantissa * 2^exponent, mantissa in [0.5,1)
    q = int(round(mantissa * (1 << 31)))
    if q == (1 << 31):
        q //= 2
        exponent += 1
    shift = 31 - exponent
    if shift < 0:
        q <<= (-shift)
        shift = 0
    return int(q), int(shift)

# helper types used in headers (names only; arrays use ap_int in C++)
def _sum_weights_per_out(W_int8: np.ndarray) -> np.ndarray:
    # W_int8 shape: [OUT_CH, IN_CH, K]
    # returns int32 sums per OUT_CH
    return W_int8.reshape(W_int8.shape[0], -1).sum(axis=1).astype(np.int32)

def _bias_int32_vector(b_f: np.ndarray, s_in: float, s_w: float, out_ch: int) -> np.ndarray:
    # Quantize bias or return zeros if no bias provided
    if b_f is None:
        return np.zeros(out_ch, dtype=np.int32)
    s_bias = s_in * s_w if (s_in and s_w) else 1.0
    return np.round(b_f / s_bias).astype(np.int32)


def _qt_weight_int8_per_tensor(W_f: np.ndarray, scale: float, zero_point: int = 0):
    Wq = np.round(W_f / scale) + zero_point
    return np.clip(Wq, -128, 127).astype(np.int8)

def _bias_int32_from_float(b_f: np.ndarray, s_in: float, s_w: float):
    s_bias = s_in * s_w
    if s_bias == 0.0: s_bias = 1.0
    bq = np.round(b_f / s_bias).astype(np.int32)
    return bq

def _as1(x):
    if isinstance(x, (list, tuple)):
        return int(x[0])
    return int(x)

def _guard_out_scale(name: str, out_scale: float):
    if out_scale is None or not np.isfinite(out_scale) or out_scale <= 0.0:
        raise ValueError(f"{name}: invalid out_scale={out_scale}")

def _guard_in_scale(name: str, in_scale: float):
    if in_scale is None or not np.isfinite(in_scale) or in_scale <= 0.0:
        raise ValueError(f"{name}: invalid in_scale={in_scale}")

def _guard_weight_scale(name: str, w_scale: float):
    if w_scale is None or not np.isfinite(w_scale) or w_scale <= 0.0:
        raise ValueError(f"{name}: invalid weight_scale={w_scale}")

def _id_guard_macro(base: str):
    return base.upper().replace('/', '_').replace('.', '_')

def _sym(name: str):
    """C identifier from layer name (keep as-is but safe for C)."""
    return name.replace('/', '_').replace('.', '_')

def _fmt_int_list(vals, per_line=16):
    out = []
    line = []
    for i, v in enumerate(vals):
        line.append(str(int(v)))
        if (i + 1) % per_line == 0:
            out.append(", ".join(line))
            line = []
    if line:
        out.append(", ".join(line))
    return ",\n    ".join(out)

def _fmt_array_2d(arr2d):
    rows = []
    for r in arr2d:
        rows.append("{ " + _fmt_int_list(r) + " }")
    return "{\n  " + ",\n  ".join(rows) + "\n}"

def _fmt_array_3d(arr3d):
    blocks = []
    for b in arr3d:
        blocks.append(_fmt_array_2d(b))
    return "{\n" + ",\n".join(blocks) + "\n}"

# -----------------------
# Emitters
# -----------------------

def _emit_header_open(fp, guard, ns="hls4csnn1d_cblk_sd"):
    fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
    fp.write("#include <ap_int.h>\n\n")
    fp.write("#include \"../constants24_sd.h\"\n\n")  # ✅ important
    fp.write(f"namespace {ns} {{\n\n")

def _emit_header_close(fp, guard, ns="hls4csnn1d_cblk_sd"):
    fp.write(f"}} // namespace\n#endif // {guard}\n")

def _emit_conv1d_header(path, lname, W_int8, rq_mult, rq_shift,
                        bias_int32_vec, input_zp, weight_sum_vec, m):
    # Guard: QCSNET2_CBLK1_QCONV1D_WEIGHTS_H style
    guard = _id_guard_macro(f"{_sym(lname)}_WEIGHTS_H")
    with open(path, "w") as fp:
        _emit_header_open(fp, guard)  # writes includes + namespace line

        sym   = _sym(lname)
        OC, IC, K = W_int8.shape
        stride = _as1(m.stride)

        # Structural constants (optional but handy for TBs)
        fp.write(f"const int {sym}_OUT_CH = {OC};\n")
        fp.write(f"const int {sym}_IN_CH  = {IC};\n")
        fp.write(f"const int {sym}_KERNEL_SIZE = {K};\n")
        fp.write(f"const int {sym}_STRIDE = {stride};\n\n")

        # Input ZP (INT8)
        fp.write(f"const ap_int<8> {sym}_input_zero_point = {int(input_zp)};\n\n")

        # Requantization arrays (duplicated per OUT_CH to match your API)
        fp.write(f"const ap_int<32> {sym}_scale_multiplier[{OC}] = {{\n  ")
        fp.write(_fmt_int_list([rq_mult]*OC))
        fp.write("\n};\n\n")

        fp.write(f"const int {sym}_right_shift[{OC}] = {{\n  ")
        fp.write(_fmt_int_list([rq_shift]*OC))
        fp.write("\n};\n\n")

        # Bias (INT32)
        fp.write(f"const acc32_t {sym}_bias[{OC}] = {{\n  ")
        fp.write(_fmt_int_list(bias_int32_vec))
        fp.write("\n};\n\n")

        # Weight sums for asymmetric correction (INT32)
        fp.write(f"const acc32_t {sym}_weight_sum[{OC}] = {{\n  ")
        fp.write(_fmt_int_list(weight_sum_vec))
        fp.write("\n};\n\n")

        # Weights (INT8): [OUT_CH][IN_CH][K]
        fp.write(f"const ap_int<8> {sym}_weights[{OC}][{IC}][{K}] = ")
        fp.write(_fmt_array_3d(W_int8))
        fp.write(";\n\n")

        _emit_header_close(fp, guard)  # closes namespace + guard


def _emit_linear_header(path, lname,
                           W_int8,                 # [OUT][IN] int8
                           rq_mult, rq_shift,      # per-tensor constants, repeated per OUT
                           bias_int32_vec,         # [OUT] int32
                           input_zp,               # int (will be placed as ap_int<8>)
                           weight_sum_vec):        # [OUT] int32
    guard = _id_guard_macro(f"{_sym(lname)}_WEIGHTS_H")
    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("#include <hls_stream.h>\n#include <ap_int.h>\n#include \"../constants24_sd.h\"\n\n")
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")

        sym = _sym(lname)
        OUT, IN = W_int8.shape

        # Structural (optional helpers)
        fp.write(f"const int {sym}_OUTPUT_SIZE = {OUT};\n")
        fp.write(f"const int {sym}_INPUT_SIZE  = {IN};\n\n")

        # Input zero-point (matches template type)
        fp.write(f"const ap_int<8> {sym}_input_zero_point = {int(input_zp)};\n\n")

        # Requant arrays (match template types; repeat the per-tensor constants)
        fp.write(f"const ap_int<32> {sym}_scale_multiplier[{OUT}] = {{\n  ")
        fp.write(_fmt_int_list([rq_mult] * OUT))
        fp.write("\n};\n\n")

        fp.write(f"const int {sym}_right_shift[{OUT}] = {{\n  ")
        fp.write(_fmt_int_list([rq_shift] * OUT))
        fp.write("\n};\n\n")

        # Bias and weight_sum (acc domain)
        fp.write(f"using acc32_t = ap_int<32>;\n")
        fp.write(f"const acc32_t {sym}_bias[{OUT}] = {{\n  ")
        fp.write(_fmt_int_list(bias_int32_vec))
        fp.write("\n};\n\n")

        fp.write(f"const acc32_t {sym}_weight_sum[{OUT}] = {{\n  ")
        fp.write(_fmt_int_list(weight_sum_vec))
        fp.write("\n};\n\n")

        # Weights (use ap_int8_c to match your template)
        fp.write(f"const ap_int8_c {sym}_weights[{OUT}][{IN}] = ")
        fp.write(_fmt_array_2d(W_int8))
        fp.write(";\n\n")

        fp.write("} // namespace hls4csnn1d_cblk_sd\n")
        fp.write(f"#endif // {guard}\n")



def _emit_bn_header(path, lname, w_q, b32, mult_arr, shift_arr):
    guard = _id_guard_macro(f"{_sym(lname)}_BN_H")
    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("#include <hls_stream.h>\n#include <ap_int.h>\n#include \"../constants24_sd.h\"\n\n")
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")
        sym = _sym(lname); C = len(w_q)

        fp.write(f"const int {sym}_C = {C};\n\n")

        fp.write(f"const ap_int8_c {sym}_weight[{C}] = {{\n  {_fmt_int_list(w_q)}\n}};\n\n")
        fp.write(f"const ap_int<32> {sym}_bias[{C}] = {{\n  {_fmt_int_list(b32)}\n}};\n\n")
        fp.write(f"const ap_int<32> {sym}_scale_multiplier[{C}] = {{\n  {_fmt_int_list(mult_arr)}\n}};\n\n")
        fp.write(f"const int {sym}_right_shift[{C}] = {{\n  {_fmt_int_list(shift_arr)}\n}};\n\n")

        fp.write("} // namespace hls4csnn1d_cblk_sd\n")
        fp.write(f"#endif // {guard}\n")



def _emit_lif_header_scalar_sd(path, lname, beta_q, theta_q, scale_q, frac_bits):
    guard = _id_guard_macro(f"{_sym(lname)}_LIF_H")
    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("#include <hls_stream.h>\n#include <ap_int.h>\n#include \"../constants24_sd.h\"\n\n")
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")
        sym = _sym(lname)
        fp.write(f"enum {{ {sym}_FRAC_BITS = {int(frac_bits)} }};\n")
        fp.write(f"const ap_int<16> {sym}_beta_int   = {int(beta_q)};\n")
        fp.write(f"const ap_int<16> {sym}_theta_int  = {int(theta_q)};\n")
        fp.write(f"const ap_int<16> {sym}_scale_int  = {int(scale_q)};\n\n")
        fp.write("} // namespace hls4csnn1d_cblk_sd\n")
        fp.write(f"#endif // {guard}\n")


def _emit_lif_header_vector_sd(path, lname, beta_arr_q, theta_arr_q, scale_q, frac_bits):
    guard = _id_guard_macro(f"{_sym(lname)}_LIF_H")
    sym   = _sym(lname)
    N     = len(beta_arr_q)
    if len(theta_arr_q) != N:
        raise ValueError("beta/theta array lengths must match")

    def _fmt_list(vals, per_line=16):
        rows = []
        for i in range(0, len(vals), per_line):
            chunk = ", ".join(str(int(v)) for v in vals[i:i+per_line])
            rows.append("    " + chunk)
        return "{\n" + ",\n".join(rows) + "\n}"

    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("#include <hls_stream.h>\n#include <ap_int.h>\n#include \"../constants24_sd.h\"\n\n")
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")
        fp.write(f"enum {{ {sym}_FRAC_BITS = {int(frac_bits)}, {sym}_OUT_CH = {int(N)} }};\n")
        fp.write(f"const ap_int<16> {sym}_scale_int = {int(scale_q)};\n\n")
        fp.write(f"const ap_int<16> {sym}_beta_int[{sym}_OUT_CH] = "  + _fmt_list(beta_arr_q)  + ";\n\n")
        fp.write(f"const ap_int<16> {sym}_theta_int[{sym}_OUT_CH] = " + _fmt_list(theta_arr_q) + ";\n\n")
        fp.write("} // namespace hls4csnn1d_cblk_sd\n")
        fp.write(f"#endif // {guard}\n")



INT16_MIN, INT16_MAX = -32768, 32767

def _to_q_i16(x: float, Q: int) -> int:
    return int(np.clip(round(float(x) * Q), INT16_MIN, INT16_MAX))

def _tensor_to_q_i16_list(t: torch.Tensor, Q: int):
    flat = t.detach().float().reshape(-1).cpu().tolist()
    return [_to_q_i16(v, Q) for v in flat]


_Q_SCALE = 1 << 12   # e.g., FRAC_BITS = 12

def _emit_qparams_header(path, lname, bit_w, scale, zp):
    guard = _id_guard_macro(f"QPARAMS_{_sym(lname)}_H")
    sym = _sym(lname)
    sym_base = sym  # no renaming, no suffix stripping

    # Q-encode the activation scale for HLS QuantIdentity (uses _Q_SCALE; not emitted)
    act_scale_int = _to_q_i16(float(scale), _Q_SCALE)

    with open(path, "w") as fp:
        _emit_header_open(fp, guard)  # must include <ap_int.h> and open your namespace

        fp.write("// Activation quantization parameters (optional for kernels)\n")
        fp.write(f"const int   {sym}_bit_width = {int(bit_w)};\n")
        fp.write(f"// const float {sym}_scale     = {float(scale):.10g};  // kept for reference only\n")
        fp.write(f"const ap_int<16> {sym_base}_act_scale_int = {int(act_scale_int)};\n")
        fp.write(f"const int   {sym}_zero_point= {int(zp)};\n\n")

        _emit_header_close(fp, guard)  # close namespace and guard


# Add this function to your weight extraction code (after the other _emit_* functions)

def _emit_rr_stats_header(path, rr_stats, input_scale=None):
    """
    Emit header with RR normalization statistics from training data.
    
    Args:
        path: Output header file path
        rr_stats: Dict with "mean" and "std" arrays (each length 4)
        input_scale: Optional input quantization scale (for reference)
    """
    guard = "RR_NORMALIZATION_STATS_H"
    
    rr_mean = rr_stats["mean"]
    rr_std = rr_stats["std"]
    
    # Precompute 1/std for faster runtime (multiply instead of divide)
    rr_std_inv = 1.0 / np.maximum(rr_std, 1e-8)
    
    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("/**\n")
        fp.write(" * RR Feature Normalization Statistics\n")
        fp.write(" * \n")
        fp.write(" * These values are computed from the TRAINING set and must be used\n")
        fp.write(" * for both training and test/inference to avoid data leakage.\n")
        fp.write(" * \n")
        fp.write(" * Normalization formula:\n")
        fp.write(" *   rr_normalized[i] = (rr_raw[i] - RR_MEAN[i]) / RR_STD[i]\n")
        fp.write(" *                    = (rr_raw[i] - RR_MEAN[i]) * RR_STD_INV[i]\n")
        fp.write(" * \n")
        fp.write(" * Input layout (184 features):\n")
        fp.write(" *   [0-179]:   ECG beat samples (already z-scored per beat in CSV)\n")
        fp.write(" *   [180-183]: RR interval features (need normalization)\n")
        fp.write(" */\n\n")
        
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")
        
        fp.write("// RR feature indices\n")
        fp.write("static const int RR_FEATURE_START = 180;\n")
        fp.write("static const int RR_FEATURE_COUNT = 4;\n\n")
        
        # Mean values
        fp.write("// Training set mean for each RR feature\n")
        fp.write("static const float RR_MEAN[4] = {\n")
        fp.write(f"    {rr_mean[0]:.10f}f,   // RR_prev\n")
        fp.write(f"    {rr_mean[1]:.10f}f,   // RR_next\n")
        fp.write(f"    {rr_mean[2]:.10f}f,   // RR_ratio\n")
        fp.write(f"    {rr_mean[3]:.10f}f    // RR_diff\n")
        fp.write("};\n\n")
        
        # Std values
        fp.write("// Training set std for each RR feature\n")
        fp.write("static const float RR_STD[4] = {\n")
        fp.write(f"    {rr_std[0]:.10f}f,   // RR_prev\n")
        fp.write(f"    {rr_std[1]:.10f}f,   // RR_next\n")
        fp.write(f"    {rr_std[2]:.10f}f,   // RR_ratio\n")
        fp.write(f"    {rr_std[3]:.10f}f    // RR_diff\n")
        fp.write("};\n\n")
        
        # Precomputed 1/std for faster runtime
        fp.write("// Precomputed 1/std for faster computation (multiply instead of divide)\n")
        fp.write("static const float RR_STD_INV[4] = {\n")
        fp.write(f"    {rr_std_inv[0]:.10f}f,   // 1/RR_prev_std\n")
        fp.write(f"    {rr_std_inv[1]:.10f}f,   // 1/RR_next_std\n")
        fp.write(f"    {rr_std_inv[2]:.10f}f,   // 1/RR_ratio_std\n")
        fp.write(f"    {rr_std_inv[3]:.10f}f    // 1/RR_diff_std\n")
        fp.write("};\n\n")
        
        # Input scale (if provided)
        if input_scale is not None:
            fp.write(f"// Input quantization scale (from QuantIdentity)\n")
            fp.write(f"static const float RR_INPUT_SCALE = {float(input_scale):.10f}f;\n")
            fp.write(f"static const float RR_INPUT_SCALE_INV = {1.0/float(input_scale):.10f}f;\n\n")
        
        fp.write("} // namespace hls4csnn1d_cblk_sd\n\n")
        fp.write(f"#endif // {guard}\n")
    
    print(f"[emit] RR stats header written to: {path}")


# -----------------------
# Orchestrator
# -----------------------

def emit_headers_for_model(model: torch.nn.Module,
                           example_input: torch.Tensor,
                           out_dir: str = "headers_int",
                           lif_frac_bits: int = 12):
    model.eval()
    _ensure_dir(out_dir)

    # Run one dry forward to initialize any lazy buffers (ignore output tuples)
    with torch.no_grad():
        try:
            _ = model(example_input)
        except Exception:
            pass

    # Track current activation qparams (propagated as in your graph)
    current_act = {"bit_width": 8, "scale": 1.0, "zero_point": 0}
    
    last_out_ch = None  # initialize outside the loop

    for name, m in model.named_modules():
        if m is model:
            continue

        # QuantIdentity (export activation qparams as optional header)
        if isinstance(m, qnn.QuantIdentity):
            aq = getattr(m, 'act_quant', getattr(m, 'output_quant', None))
            bit_w, s, zp, _ = _get_bit_scale_zp_from_quant(aq)
            _emit_qparams_header(os.path.join(out_dir, f"qparams_{name}.h"), name, bit_w, s, zp)
            current_act = {"bit_width": bit_w, "scale": s, "zero_point": zp}
            continue


        if isinstance(m, qnn.QuantConv1d):
            # Float → INT8 weights
            Wf = m.weight.detach().cpu().numpy()            # [OUT, IN, K]
            wq = getattr(m, 'weight_quant', None)
            wb, s_w, z_w, _ = _get_bit_scale_zp_from_quant(wq)
        
            _guard_in_scale(name, current_act['scale'])
            _guard_weight_scale(name, s_w)
        
            W_int8 = _qt_weight_int8_per_tensor(Wf, s_w, z_w)
        
            # Output activation qparams (for requant)
            oq = getattr(m, 'output_quant', None)
            ob, s_out, z_out, _ = _get_bit_scale_zp_from_quant(oq)
            _guard_out_scale(name, s_out)
        
            # Integer requant constants (per-tensor → repeat per OUT_CH)
            M = (current_act['scale'] * s_w) / s_out
            rq_mult, rq_shift = _quantize_multiplier(M)
        
            # Bias (if present) → INT32 vector (length OUT_CH); else zeros
            b_f = m.bias.detach().cpu().numpy() if (hasattr(m, 'bias') and m.bias is not None) else None
            bias_int32_vec = _bias_int32_vector(b_f, current_act['scale'], s_w, W_int8.shape[0])
        
            # Weight sums per output channel (for asymmetric correction)
            weight_sum_vec = _sum_weights_per_out(W_int8)
        
            # Input zero point (INT8) for asymmetric correction
            input_zp = current_act['zero_point']
        
            # Emit header that matches your Conv1D_SD::forward signature
            _emit_conv1d_header(
                os.path.join(out_dir, f"{name}_weights.h"),
                name,
                W_int8,
                rq_mult, rq_shift,
                bias_int32_vec,
                input_zp,
                weight_sum_vec,
                m
            )
        
            # Advance activation qparams for the next layer
            current_act = {"bit_width": ob, "scale": s_out, "zero_point": z_out}
            continue


        # BN -> ScaleBias (for BatchNorm1dToQuantScaleBias)
        if hasattr(qnn, 'BatchNorm1dToQuantScaleBias') and isinstance(m, qnn.BatchNorm1dToQuantScaleBias):
            # gamma, beta (float, per channel)
            gamma = _to_one(getattr(m, 'weight', None)) if getattr(m, 'weight', None) is not None else _to_one(getattr(m, 'scale', None))
            beta  = _to_one(getattr(m, 'bias',   None)) if getattr(m, 'bias',   None) is not None else _to_one(getattr(m, 'beta',  None))
            if gamma is None: gamma = 1.0
            if beta  is None: beta  = 0.0
            gamma = np.array(gamma, dtype=np.float32).reshape(-1)
            beta  = np.array(beta,  dtype=np.float32).reshape(-1)
            C = gamma.shape[0]
        
            # Input/output quant
            s_in = float(current_act['scale']);  z_in = int(current_act['zero_point']);  _guard_in_scale(name, s_in)
            oq   = getattr(m, 'output_quant', None)
            _, s_out, _, _ = _get_bit_scale_zp_from_quant(oq);  _guard_out_scale(name, s_out)
        
            # Brevitas BN quantized weight
            qW = m.quant_weight()  # IntQuantTensor
            w_q = qW.int().detach().cpu().numpy().astype(np.int8)    # [C]
            s_w = float(_to_one(qW.scale))                            # scalar
        
            # Shared requant scale S and its integer pair
            S = s_w * (s_in / s_out)
            mult_S, shift_S = _quantize_multiplier(S)
        
            # Int32 bias per channel (no int8 clipping)
            # M_c = (s_in/s_out) * gamma_c  = S * w_q[c]  (approximately)
            M = gamma * (s_in / s_out)                          # [C]
            b32 = np.round((beta / s_out - M * z_in) / S).astype(np.int32)  # [C]
        
            _emit_bn_header(
                os.path.join(out_dir, f"{name}_bn.h"),
                name,
                w_q.tolist(),
                b32.tolist(),
                [mult_S] * C,
                [shift_S] * C
            )
        
            # Advance activation qparams
            current_act = {"bit_width": current_act['bit_width'], "scale": s_out, "zero_point": 0}
            continue



        # QuantLinear
        if isinstance(m, qnn.QuantLinear):
            # Float → INT8 weights
            Wf = m.weight.detach().cpu().numpy()             # [OUT, IN]
            wq = getattr(m, 'weight_quant', None)
            wb, s_w, z_w, _ = _get_bit_scale_zp_from_quant(wq)
        
            _guard_in_scale(name, current_act['scale'])
            _guard_weight_scale(name, s_w)
        
            W_int8 = _qt_weight_int8_per_tensor(Wf, s_w, z_w)
        
            # Output activation qparams (for requant)
            oq = getattr(m, 'output_quant', None)
            ob, s_out, z_out, _ = _get_bit_scale_zp_from_quant(oq)
            _guard_out_scale(name, s_out)
        
            # Requant: M = (s_in * s_w) / s_out
            M = (current_act['scale'] * s_w) / s_out
            rq_mult, rq_shift = _quantize_multiplier(M)
        
            # Bias (if present) → INT32; else zeros
            b_f = m.bias.detach().cpu().numpy() if (hasattr(m, 'bias') and m.bias is not None) else None
            bias_int32_vec = _bias_int32_vector(b_f, current_act['scale'], s_w, W_int8.shape[0])
        
            # Weight sums for asymmetric correction: sum over input dim
            weight_sum_vec = W_int8.sum(axis=1).astype(np.int32)
        
            # Input zero-point for asymmetric correction
            input_zp = current_act['zero_point']
        
            # Emit header that matches Linear1D_SD::forward
            _emit_linear_header(
                os.path.join(out_dir, f"{name}_weights.h"),
                name,
                W_int8,
                rq_mult, rq_shift,
                bias_int32_vec,
                input_zp,
                weight_sum_vec
            )
        
            # Advance activation qparams
            current_act = {"bit_width": ob, "scale": s_out, "zero_point": z_out}
            continue

        
        if isinstance(m, snn.Leaky):
            scale_in = float(current_act['scale'])
            Q = 1 << lif_frac_bits
        
            # Grab beta/threshold as tensors (handles both scalar and vector)
            beta_t  = m.beta if isinstance(m.beta, torch.Tensor) else torch.as_tensor(m.beta)
            thr_t   = m.threshold if isinstance(m.threshold, torch.Tensor) else torch.as_tensor(m.threshold)
        
            beta_t  = beta_t.detach().float()
            print("beta: ", beta_t)
            thr_t   = thr_t.detach().float()
        
            # Number of “neurons” (channels) from parameter size
            n_beta  = int(beta_t.numel())
            n_thr   = int(thr_t.numel())
            if n_beta != n_thr and n_beta != 1 and n_thr != 1:
                raise ValueError(f"LIF param size mismatch: beta has {n_beta}, threshold has {n_thr}")
        
            # Broadcast if one is scalar
            out_ch = max(n_beta, n_thr)
            if n_beta == 1 and out_ch > 1:
                beta_t = beta_t.expand(out_ch)
            if n_thr == 1 and out_ch > 1:
                thr_t = thr_t.expand(out_ch)
        
            # Quantize
            beta_arr_q  = _tensor_to_q_i16_list(beta_t, Q)
            theta_arr_q = _tensor_to_q_i16_list(thr_t, Q)
            scale_q     = _to_q_i16(scale_in, Q)   # keep scale scalar for now
        
            # Emit vector or scalar header depending on out_ch
            out_path = os.path.join(out_dir, f"{name}_lif.h")
            if out_ch > 1:
                _emit_lif_header_vector_sd(
                    out_path, name, beta_arr_q, theta_arr_q, scale_q, lif_frac_bits
                )
            else:
                _emit_lif_header_scalar_sd(
                    out_path, name, beta_arr_q[0], theta_arr_q[0], scale_q, lif_frac_bits
                )
        
            # LIF outputs binary spikes {0,1} → treat next op input scale as 1.0
            current_act = {"bit_width": 8, "scale": 1.0, "zero_point": 0}
            continue
            
        # =====================================================================
        # NEW: Emit RR stats header if provided
        # =====================================================================
        if rr_stats is not None:
            _emit_rr_stats_header(
                os.path.join(out_dir, "rr_normalization_stats.h"),
                rr_stats,
            )

    print(f"[emit] C++ headers written to: {os.path.abspath(out_dir)}")


In [ ]:
def emit_headers_for_qcsnn24_dict(
    model_dict, 
    out_dir="smote_with_stage2_rr_features_headers_int24", 
    lif_frac_bits=12,
    trunk_size=480,
    rr_dim=4,
    rr_to_stage1=True,
    rr_to_stage2=True,
    rr_stats=None,
):
    """
    Export net24, net2, net4 into the SAME output folder.
    
    Ablation configurations:
      - Ablation 1: rr_to_stage1=True,  rr_to_stage2=False  (RR → S1 only)
      - Ablation 2: rr_to_stage1=False, rr_to_stage2=True   (RR → S2 only)
      - Ablation 3: rr_to_stage1=True,  rr_to_stage2=True   (RR → Both)
      - No RR:      rr_to_stage1=False, rr_to_stage2=False
    """
    _ensure_dir(out_dir)
    x_trunk = torch.randn(1, 1, 180)
    stage1_input = trunk_size + rr_dim if rr_to_stage1 else trunk_size
    x_head2 = torch.randn(1, stage1_input)
    stage2_input = trunk_size + rr_dim if rr_to_stage2 else trunk_size
    x_head4 = torch.randn(1, stage2_input)
    
    print(f"Exporting weights:")
    print(f"  net24 input: (1, 1, 180)")
    print(f"  net2 (Stage 1) input: (1, {stage1_input}) [RR: {rr_to_stage1}]")
    print(f"  net4 (Stage 2) input: (1, {stage2_input}) [RR: {rr_to_stage2}]")
    
    emit_headers_for_model(model_dict["net24"], x_trunk, out_dir=out_dir, lif_frac_bits=lif_frac_bits)
    emit_headers_for_model(model_dict["net2"],  x_head2, out_dir=out_dir, lif_frac_bits=lif_frac_bits)
    emit_headers_for_model(model_dict["net4"],  x_head4, out_dir=out_dir, lif_frac_bits=lif_frac_bits)
    
    if rr_stats is not None:
        _emit_rr_stats_header(os.path.join(out_dir, "rr_stats.h"), rr_stats)
    
    print(f"[emit] C++ headers written to: {os.path.abspath(out_dir)}")

In [ ]:
# model_final is already a dict: {"net24","net2","net4"}

# safest: move to CPU before extraction (avoids device mismatches)
model_export = {
    "net24": model_final["net24"].cpu().eval(),
    "net2":  model_final["net2"].cpu().eval(),
    "net4":  model_final["net4"].cpu().eval(),
}

# Ablation 1: RR → Stage 1 only (THE WINNER)
emit_headers_for_qcsnn24_dict(
    model_dict=model_export,
    out_dir="weights_sd/intra_patient/ablation3_smote_rr_stage12",
    rr_to_stage1=True,
    rr_to_stage2=True,
    rr_stats=rr_stats  # <-- Pass your rr_stats here
)

In [ ]:
import time
import subprocess
import numpy as np
import torch

def get_gpu_power_w():
    """GPU board power (Watts) via nvidia-smi. Returns None if unavailable."""
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=power.draw", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=False
        )
        s = r.stdout.strip()
        return float(s) if s else None
    except Exception:
        return None


@torch.no_grad()
def evaluate_two_stage_optionA(
    model, dataloader, device, num_steps=10,
    signal_len=180,                                    # ADDED
    rr_dim=4,                                          # ADDED
    class_names2=("Normal(0)", "Abnormal(1)"),
    class_names4=("N(0)", "SVEB(1)", "VEB(2)", "F(3)"),
    verbose=True,
    measure_power=True,
    power_sample_every_batches=10,
):
    # Ensure modules on device + eval
    model["net24"] = model["net24"].to(device).eval()
    model["net2"]  = model["net2"].to(device).eval()
    model["net4"]  = model["net4"].to(device).eval()

    tp2 = [0, 0]; fp2 = [0, 0]; tn2 = [0, 0]; fn2 = [0, 0]
    tp4 = [0, 0, 0, 0]; fp4 = [0, 0, 0, 0]; tn4 = [0, 0, 0, 0]; fn4 = [0, 0, 0, 0]

    n = 0
    correct_final = 0
    correct_stage1 = 0
    routed_total = 0

    total_t0 = time.perf_counter()
    model_time_s = 0.0
    power_samples_w = []

    for b, (xb, y4) in enumerate(dataloader):
        if measure_power and (b % power_sample_every_batches == 0) and (device.type == "cuda"):
            p = get_gpu_power_w()
            if p is not None:
                power_samples_w.append(p)

        xb = xb.to(device, non_blocking=True)
        y4 = y4.to(device, non_blocking=True).long()
        y2 = (y4 > 0).long()

        B = xb.size(0)
        n += B

        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        # FIXED: pass signal_len and rr_dim
        pred2, final4, routed_mask = forward_two_stage_rr_both(
            model, num_steps, xb,
            gate="argmax",
            signal_len=signal_len,
            rr_dim=rr_dim
        )

        if device.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        model_time_s += (t1 - t0)

        routed_total += int(routed_mask.sum().item())
        correct_stage1 += (pred2 == y2).sum().item()
        correct_final  += (final4 == y4).sum().item()

        for c in (0, 1):
            tp2[c] += ((pred2 == c) & (y2 == c)).sum().item()
            fp2[c] += ((pred2 == c) & (y2 != c)).sum().item()
            tn2[c] += ((pred2 != c) & (y2 != c)).sum().item()
            fn2[c] += ((pred2 != c) & (y2 == c)).sum().item()

        for c in (0, 1, 2, 3):
            tp4[c] += ((final4 == c) & (y4 == c)).sum().item()
            fp4[c] += ((final4 == c) & (y4 != c)).sum().item()
            tn4[c] += ((final4 != c) & (y4 != c)).sum().item()
            fn4[c] += ((final4 != c) & (y4 == c)).sum().item()

    total_time_s = time.perf_counter() - total_t0

    stage1_acc = 100.0 * correct_stage1 / max(n, 1)
    final_acc  = 100.0 * correct_final  / max(n, 1)
    routed_pct = 100.0 * routed_total   / max(n, 1)

    m2 = {"acc": stage1_acc, **_per_class_metrics(tp2, fp2, tn2, fn2, class_names=list(class_names2))}
    m4 = {"acc": final_acc,  **_per_class_metrics(tp4, fp4, tn4, fn4, class_names=list(class_names4))}

    perf = {
        "total_samples": int(n),
        "routed_pct": float(routed_pct),
        "model_time_s": float(model_time_s),
        "time_per_inference_ms_model_only": float((model_time_s / max(n, 1)) * 1e3),
        "throughput_sps_model_only": float(n / max(model_time_s, 1e-12)),
        "total_time_s": float(total_time_s),
        "time_per_inference_ms_total": float((total_time_s / max(n, 1)) * 1e3),
        "throughput_sps_total": float(n / max(total_time_s, 1e-12)),
    }

    if device.type == "cuda" and power_samples_w:
        avg_power_w = float(np.mean(power_samples_w))
        perf["avg_gpu_power_w"] = avg_power_w
        perf["energy_per_inference_mJ_model_only"] = float(avg_power_w * (model_time_s / max(n, 1)) * 1e3)
        perf["energy_per_inference_mJ_total"] = float(avg_power_w * (total_time_s / max(n, 1)) * 1e3)
    else:
        perf["avg_gpu_power_w"] = None
        perf["energy_per_inference_mJ_model_only"] = None
        perf["energy_per_inference_mJ_total"] = None

    if verbose:
        print("\n==================== TWO-STAGE OPTION-A EVAL ====================")
        print(f"Routed % (sent to net4): {routed_pct:.2f}%")

        print("\n--- Stage-1 Binary Metrics (per-class) ---")
        print(f"Stage-1 Acc: {m2['acc']:.2f}% | Macro F1: {m2['f1_macro']:.4f}")
        for cname, v in m2["per_class"].items():
            print(f"{cname:>15s} | Prec={v['precision']:.4f}  Rec={v['recall']:.4f}  F1={v['f1']:.4f}")

        print("\n--- Final Two-Stage 4-Class Metrics (per-class) ---")
        print(f"Final Acc : {m4['acc']:.2f}% | Macro F1: {m4['f1_macro']:.4f}")
        for cname, v in m4["per_class"].items():
            print(f"{cname:>15s} | Prec={v['precision']:.4f}  Rec={v['recall']:.4f}  F1={v['f1']:.4f}")

        print("\n--- Performance / Energy (estimates) ---")
        print(f"Model-only time/inference: {perf['time_per_inference_ms_model_only']:.4f} ms"
              f" | Throughput: {perf['throughput_sps_model_only']:.2f} samples/s")
        print(f"Total time/inference     : {perf['time_per_inference_ms_total']:.4f} ms"
              f" | Throughput: {perf['throughput_sps_total']:.2f} samples/s")
        if perf["avg_gpu_power_w"] is not None:
            print(f"Avg GPU power (sampled)  : {perf['avg_gpu_power_w']:.2f} W")
            print(f"Energy/inf (model-only)  : {perf['energy_per_inference_mJ_model_only']:.4f} mJ")
            print(f"Energy/inf (total)       : {perf['energy_per_inference_mJ_total']:.4f} mJ")
        else:
            print("Avg GPU power (sampled)  : N/A (CPU run or nvidia-smi unavailable)")

        print("=================================================================\n")

    return m2, m4, routed_pct, perf

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m2, m4, routed_pct, perf = evaluate_two_stage_optionA(
    model_export, test_loader, device, 
    num_steps=10, 
    signal_len=180,  # ADD
    rr_dim=4,        # ADD
    verbose=True,
    measure_power=True, 
    power_sample_every_batches=10
)
print(perf)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, SubsetRandomSampler

import snntorch as snn
from snntorch import spikegen, utils, surrogate
from snntorch.functional import quant

import brevitas.nn as qnn
# from brevitas.nn import QuantConv1d, QuantIdentity, QuantLinear, BatchNorm1dToQuantScaleBias
from brevitas.quant import Int8WeightPerTensorFloat, Int8ActPerTensorFloat, Int32Bias
from brevitas.export import export_qonnx
from brevitas.core.scaling      import ScalingImplType
from brevitas.core.restrict_val import RestrictValueType
from brevitas.core.quant        import QuantType


from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.core.datatype import DataType
#from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

import optuna

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import itertools
import csv, copy, glob
import imblearn, imblearn.over_sampling
from collections import Counter, OrderedDict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve, auc

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import label_binarize, normalize
from scipy.signal import butter, lfilter, freqz

import os, sys, time, datetime, argparse, json


/home/velox-217533/anaconda3/envs/fau_snn_torch-cuda/lib/python3.12/site-packages/brevitas/graph/equalize.py:69: UserWarning: fast_hadamard_transform package not found, using standard pytorch kernels
  warnings.warn("fast_hadamard_transform package not found, using standard pytorch kernels")
/home/velox-217533/anaconda3/envs/fau_snn_torch-cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os, glob
import numpy as np
import pandas as pd
import torch
from collections import Counter

def load_train_test_data(
    mitbih_path='/home/velox-217533/Projects/fau_projects/research/data/mitbih_processed_intra_patient_4class_180_center90_rr_features_filtered/train',
    folders=None,
    random_state=42,
    max_files_per_folder=None,
    rr_cols=4,
    beat_len=180,
    rr_stats=None,              # pass rr_stats here for val/test to avoid leakage
    return_rr_stats=False,      # True for train load, False for test load
    eps=1e-8,
):
    if folders is None:
        folders = ['normal', 'sveb', 'veb', 'f']
    label_mapping = {'normal': 0, 'sveb': 1, 'veb': 2, 'f': 3}

    expected_cols = beat_len + rr_cols  # 184

    X_parts, y_parts = [], []

    for folder in folders:
        folder_path = os.path.join(mitbih_path, folder)
        csvs = sorted(
            glob.glob(os.path.join(folder_path, '*.csv')) +
            glob.glob(os.path.join(folder_path, '*.CSV'))
        )
        if not csvs:
            print(f'Warning: no CSVs found in: {folder_path}')
            continue

        if max_files_per_folder is not None:
            csvs = csvs[:max_files_per_folder]

        for fpath in csvs:
            df = pd.read_csv(
                fpath, dtype=np.float32, engine='c',
                usecols=lambda c: c != 'Unnamed: 0'
            )
            if df.shape[0] == 0:
                df = pd.read_csv(fpath, header=None, dtype=np.float32, engine='c')

            df = df.dropna(axis=1, how='all')
            arr = df.to_numpy(copy=False)

            if arr.shape[1] != expected_cols:
                raise ValueError(
                    f'Expected {expected_cols} cols (180+4). '
                    f'Got {arr.shape[1]} in {fpath}'
                )

            X_parts.append(arr)
            y_parts.extend([label_mapping[folder]] * arr.shape[0])

    if not X_parts:
        raise FileNotFoundError(f'No usable CSV rows under {mitbih_path}')

    X = np.vstack(X_parts).astype(np.float32, copy=False)
    y = np.asarray(y_parts, dtype=np.int64)

    # shuffle
    rng = np.random.RandomState(random_state)
    perm = rng.permutation(len(y))
    X = X[perm]
    y = y[perm]

    # split beat + rr
    X_beat = X[:, :beat_len]                  # (N,180) already z-scored per beat
    X_rr   = X[:, beat_len:beat_len+rr_cols]  # (N,4)   normalize separately

    # sanitize
    X_beat = np.nan_to_num(X_beat, nan=0.0).astype(np.float32, copy=False)
    X_rr   = np.nan_to_num(X_rr,   nan=0.0).astype(np.float32, copy=False)

    # rr normalization (train stats only)
    if rr_stats is None:
        rr_mean = X_rr.mean(axis=0)
        rr_std  = X_rr.std(axis=0)
        rr_std  = np.maximum(rr_std, eps)
        rr_stats = {"mean": rr_mean.astype(np.float32), "std": rr_std.astype(np.float32)}
    else:
        rr_mean = rr_stats["mean"]
        rr_std  = np.maximum(rr_stats["std"], eps)

    X_rr_norm = (X_rr - rr_mean) / rr_std

    # recombine into 184
    X_final = np.concatenate([X_beat, X_rr_norm], axis=1).astype(np.float32, copy=False)

    # prints separated (NO confusion)
    print(f"Loaded: {mitbih_path}  X={X_final.shape}")
    print(f"Beat(180) stats: min={X_beat.min():.2f} max={X_beat.max():.2f} mean={X_beat.mean():.4f} std={X_beat.std():.4f}")
    print(f"RR(4) raw stats : min={X_rr.min():.2f} max={X_rr.max():.2f} mean={X_rr.mean():.4f} std={X_rr.std():.4f}")
    print(f"RR(4) norm stats: min={X_rr_norm.min():.2f} max={X_rr_norm.max():.2f} mean={X_rr_norm.mean():.4f} std={X_rr_norm.std():.4f}")

    # torch tensors
    X_tensor = torch.from_numpy(X_final).unsqueeze(1)  # (N,1,184)
    y_tensor = torch.from_numpy(y)                     # (N,)

    print("Class distribution:", Counter(y_tensor.tolist()))
    print("X shape:", tuple(X_tensor.shape), "| y shape:", tuple(y_tensor.shape))

    if return_rr_stats:
        return X_tensor, y_tensor, rr_stats
    return X_tensor, y_tensor


In [3]:
import torch
from torch.utils.data import TensorDataset

def create_csnn_datasets(X_train, y_train, X_test=None, y_test=None):
    """
    Create TensorDataset objects for training/evaluation.

    Assumes:
      - X_train is a torch.Tensor of shape (N, 1, L)  (L can be 180 or 184)
      - y_train is a torch.Tensor of shape (N,)
    """
    train_dataset = TensorDataset(X_train, y_train)

    if X_test is None:
        print("train_dataset created successfully")
        return train_dataset

    test_dataset = TensorDataset(X_test, y_test)
    print("train_dataset and test_dataset created successfully")
    return train_dataset, test_dataset

In [4]:
def create_qcsnn_model24_rr_both(num_bits=8, input_size=180, rr_dim=4, stride4=1, kernel_size=3, 
                                  dropout4=0.35, beta4=0.5, slope4=25, 
                                  threshold4=0.5, learn_beta4=True, learn_threshold4=True):
    """Factory function for 4-class QCSNN — RR features routed to BOTH stages"""

    spike_grad4 = snn.surrogate.fast_sigmoid(slope=slope4)
    
    # Calculate output sizes after 3 conv blocks
    output_size1 = (input_size - kernel_size) // stride4 + 1
    output_size1 = output_size1 // 2  # MaxPool
    
    output_size2 = (output_size1 - kernel_size) // stride4 + 1
    output_size2 = output_size2 // 2  # MaxPool
    
    output_size3 = (output_size2 - kernel_size) // stride4 + 1
    output_size3 = output_size3 // 2  # MaxPool
    
    flattened_size = output_size3 * 24  # 480 when input_size=180

    print(f"Output sizes: Block1={output_size1}, Block2={output_size2}, Block3={output_size3}")
    print(f"Flattened size: {flattened_size}")
    print(f"Stage 1 input: {flattened_size + rr_dim} (trunk + RR)")
    print(f"Stage 2 input: {flattened_size + rr_dim} (trunk + RR)")  # <-- CHANGED
    
    # Backbone (unchanged)
    csnet24 = torch.nn.Sequential(OrderedDict([
        ("qcsnet24_cblk1_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
    
        ("qcsnet24_cblk1_qconv1d",
         qnn.QuantConv1d(
             1, 16, 3, stride=stride4, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
    
        ("qcsnet24_cblk1_batch_norm",
         qnn.BatchNorm1dToQuantScaleBias(
             16,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             bias_quant=Int32Bias,
             return_quant_tensor=True)),

        ("qcsnet24_cblk1_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True)),
        ("qcsnet24_cblk1_max_pool", torch.nn.MaxPool1d(2, 2)),
    
        ("qcsnet24_cblk2_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
        
        ("qcsnet24_cblk2_qconv1d",
         qnn.QuantConv1d(
             16, 16, 3, stride=stride4, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
    
        ("qcsnet24_cblk2_batch_norm",
         qnn.BatchNorm1dToQuantScaleBias(
             16,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             bias_quant=Int32Bias,
             return_quant_tensor=True)),
    
        ("qcsnet24_cblk2_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True)),
        ("qcsnet24_cblk2_max_pool", torch.nn.MaxPool1d(2, 2)),
    
        ("qcsnet24_cblk3_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
        
        ("qcsnet24_cblk3_qconv1d",
         qnn.QuantConv1d(
             16, 24, 3, stride=stride4, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
    
        ("qcsnet24_cblk3_batch_norm",
         qnn.BatchNorm1dToQuantScaleBias(
             24,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             bias_quant=Int32Bias,
             return_quant_tensor=True)),
        
        ("qcsnet24_cblk3_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True)),
        ("qcsnet24_cblk3_max_pool", torch.nn.MaxPool1d(2, 2)),
        
        ("qcsnet24_flatten", torch.nn.Flatten()),
    ]))
    
    # Stage 1 head (binary) — 484 input (unchanged)
    csnet2_head = torch.nn.Sequential(OrderedDict([
        ("qcsnet2_lblk1_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),

        ("qcsnet2_lblk1_qlinear",
         qnn.QuantLinear(
             flattened_size + rr_dim,  # 484
             2, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
        ("qcsnet2_lblk1_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True, output=True)),
    ]))

    # Stage 2 head (4-class) — NOW 484 input (CHANGED from 480)
    csnet4_head = torch.nn.Sequential(OrderedDict([
        ("qcsnet4_lblk1_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
        
        ("qcsnet4_lblk1_qlinear",
         qnn.QuantLinear(
             flattened_size + rr_dim,  # <-- CHANGED: 484 instead of 480
             128, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
        ("qcsnet4_lblk1_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True)),
    
        ("qcsnet4_lblk2_input",
         qnn.QuantIdentity(bit_width=num_bits, return_quant_tensor=True)),
        
        ("qcsnet4_lblk2_qlinear",
         qnn.QuantLinear(
             128, 4, bias=False,
             weight_bit_width=num_bits,
             weight_quant=Int8WeightPerTensorFloat,
             output_quant=Int8ActPerTensorFloat,
             return_quant_tensor=True)),
        ("qcsnet4_lblk2_leaky",
         snn.Leaky(beta=beta4, learn_beta=learn_beta4,
                   spike_grad=spike_grad4,
                   threshold=threshold4, learn_threshold=learn_threshold4,
                   init_hidden=True, output=True)),
    ]))
    
    # Fix runtime_shape for BatchNorm
    csnet24.qcsnet24_cblk1_batch_norm.runtime_shape = (1, -1, 1)
    csnet24.qcsnet24_cblk2_batch_norm.runtime_shape = (1, -1, 1)
    csnet24.qcsnet24_cblk3_batch_norm.runtime_shape = (1, -1, 1)
    
    model24 = {"net24": csnet24, 
               "net2": csnet2_head,
               "net4": csnet4_head}

    return model24

def forward_pass_rr_both(net, num_steps, data, sig_len=180, rr_dim=4):
    """Forward pass with RR features routed to BOTH stages"""
    net24 = net["net24"]
    net2 = net["net2"]
    net4 = net["net4"]

    mem2_rec, mem4_rec = [], []
    spk2_rec, spk4_rec = [], []

    utils.reset(net24)
    utils.reset(net2)
    utils.reset(net4)

    # data is (B, 1, 184)
    x_sig = data[:, :, :sig_len]                           # (B, 1, 180) -> trunk
    rr = data[:, :, sig_len:sig_len + rr_dim].squeeze(1)   # (B, 4)

    for step in range(num_steps):
        out_body = net24(x_sig)                            # (B, 480)

        # Stage 1: trunk + RR
        out2_in = torch.cat([out_body, rr], dim=1)         # (B, 484)

        # Stage 2: trunk + RR (CHANGED — was just out_body)
        out4_in = torch.cat([out_body, rr], dim=1)         # (B, 484)

        spk_out2, mem_out2 = net2(out2_in)
        spk_out4, mem_out4 = net4(out4_in)                 # <-- uses 484 now

        spk2_rec.append(spk_out2)
        mem2_rec.append(mem_out2)
        spk4_rec.append(spk_out4)
        mem4_rec.append(mem_out4)

    return torch.stack(spk2_rec), torch.stack(mem2_rec), torch.stack(spk4_rec), torch.stack(mem4_rec)

In [5]:
# load_qcsnn24_utils.py
"""
Utility functions for saving/loading QCSNN24 RR→Both model.
"""


def load_qcsnn24_dict_state(model_factory, checkpoint_path, device="cpu"):
    """
    Load model from checkpoint file.
    
    Args:
        model_factory: Function that creates the model dict (e.g., create_qcsnn_model24_rr_both)
        checkpoint_path: Path to the .pth checkpoint file
        device: Device to load model to ("cpu" or "cuda")
    
    Returns:
        model: Dictionary with {"net24", "net2", "net4"}
    """
    # Load state dict from file
    state_cpu = torch.load(checkpoint_path, map_location="cpu")
    
    # Create fresh model
    model = model_factory()  # {"net24", "net2", "net4"}
    
    # Load weights
    model["net24"].load_state_dict(state_cpu["net24"], strict=True)
    model["net2"].load_state_dict(state_cpu["net2"], strict=True)
    model["net4"].load_state_dict(state_cpu["net4"], strict=True)
    
    # Move to device and set to eval mode
    model["net24"].to(device).eval()
    model["net2"].to(device).eval()
    model["net4"].to(device).eval()
    
    print(f"[Loaded] Model from {checkpoint_path} -> {device}")
    
    return model


# ============================================================================
# Example usage
# ============================================================================
if __name__ == "__main__":
    print("""
Usage example:

    from load_qcsnn24_utils import load_qcsnn24_dict_state
    from your_model_file import create_qcsnn_model24_rr_both
    
    # Load model from checkpoint
    model24 = load_qcsnn24_dict_state(
        model_factory=create_qcsnn_model24_rr_both,
        checkpoint_path="fold3_with_stage12_rr_features_SMOTE_fulltrain_finetuned_20250304_123456.pth",
        device="cpu"  # or "cuda"
    )
    
    # Now use with dumper
    from dump_brevitas_qcsnn24_rrboth import dump_qcsnn24_rrboth
    
    results = dump_qcsnn24_rrboth(
        model=model24,
        inputs=sample,
        num_steps=10,
        out_dir="./compare_outputs/"
    )
    """)


Usage example:

    from load_qcsnn24_utils import load_qcsnn24_dict_state
    from your_model_file import create_qcsnn_model24_rr_both
    
    # Load model from checkpoint
    model24 = load_qcsnn24_dict_state(
        model_factory=create_qcsnn_model24_rr_both,
        checkpoint_path="fold3_with_stage12_rr_features_SMOTE_fulltrain_finetuned_20250304_123456.pth",
        device="cpu"  # or "cuda"
    )
    
    # Now use with dumper
    from dump_brevitas_qcsnn24_rrboth import dump_qcsnn24_rrboth
    
    results = dump_qcsnn24_rrboth(
        model=model24,
        inputs=sample,
        num_steps=10,
        out_dir="./compare_outputs/"
    )
    


In [14]:
# dump_brevitas_qcsnn24_rrboth.py
"""
Layer-by-layer dumper for QCSNN24 RR→Both two-stage model.
Outputs match the format expected by compare_layers.py and the C++ testbench.

Usage:
    model24 = create_qcsnn_model24_rr_both(...)
    model24 = load_checkpoint(model24, "checkpoint.pth")
    
    results = dump_qcsnn24_rrboth(model24, inputs, num_steps=10, out_dir="./compare_outputs/")
    
    # Then compare:
    # python compare_layers.py --dirs ./compare_outputs/ ./cpp_outputs/ \
    #     --layers input conv1 bn1 lif1 mp1 ...
"""

import os
import torch
from snntorch import utils
from typing import Dict, Any
                
# ---------- writer: matches your comparer's format ----------
@torch.no_grad()
def write_featmap_txt(tNCL: torch.Tensor, path: str):
    """
    tNCL: int8 tensor shaped (N, C, L)
    """
    tNCL = tNCL.detach().cpu().to(torch.int8)
    N, C, L = tNCL.shape
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(f"# shape: {(N, C, L)}, mode: int\n")
        for n in range(N):
            f.write(f"=== sample {n+1} ===\n")
            for c in range(C):
                row = " ".join(str(int(v)) for v in tNCL[n, c].tolist())
                f.write(f"ch {c}: {row}\n")


# ---------- quant-tensor → int8 helper ----------
def qt_to_int8_ch_first(x) -> torch.Tensor:
    """
    Accepts either a Brevitas IntQuantTensor (has .int()/.value) or a plain Tensor.
    Returns (N, C, L) int8 (adds L=1 when input is 2-D).
    """
    if hasattr(x, "int") and hasattr(x, "value"):
        xi = x.int()                           # integer view from Brevitas
    else:
        xi = torch.round(x).to(torch.int32)    # e.g., spikes or pooled floats

    if xi.ndim == 2:                           # (N, C) → (N, C, 1)
        xi = xi.unsqueeze(-1)
    if xi.ndim != 3:
        raise ValueError(f"Expected (N,C,L), got {tuple(xi.shape)}")

    return xi.to(torch.int16).clamp_(-128, 127).to(torch.int8)



# ---------- Spike to binary (for comparison with C++) ----------
def spikes_to_binary(spk: torch.Tensor) -> torch.Tensor:
    """
    Convert spike tensor to binary (0/1) for comparison with C++ output.
    C++ normalizes: >0 → 1, else → 0
    """
    return (spk > 0).to(torch.int8)


# ---------- Safe ndim accessor for Brevitas tensors ----------
def get_ndim(x):
    """Get ndim from Brevitas IntQuantTensor or plain Tensor."""
    if hasattr(x, 'value'):
        return x.value.ndim
    return x.ndim


# ---------- Safe unsqueeze for Brevitas tensors ----------
def safe_unsqueeze(x, dim=-1):
    """Unsqueeze that works with both Brevitas IntQuantTensor and plain Tensor."""
    if hasattr(x, 'value'):
        return x.value.unsqueeze(dim)
    return x.unsqueeze(dim)


# ---------- Prepare tensor for write_featmap_txt ----------
def prepare_for_write(x):
    """
    Prepare tensor for writing: extract value if Brevitas, ensure 3D.
    Returns plain torch.Tensor.
    """
    # Extract underlying tensor if Brevitas
    if hasattr(x, 'value'):
        t = x.value
    else:
        t = x
    
    # Ensure 3D: (N, C) → (N, C, 1)
    if t.ndim == 2:
        t = t.unsqueeze(-1)
    
    return t


# ---------- Main dumper for QCSNN24 RR→Both ----------
@torch.no_grad()
def dump_qcsnn24_rrboth(
    model: Dict,                   # {"net24": trunk, "net2": binary_head, "net4": multi_head}
    inputs: torch.Tensor,          # Shape: (N, 1, 184) - 180 signal + 4 RR features
    num_steps: int = 10,
    out_dir: str = "./compare_outputs/",
    sig_len: int = 180,
    rr_dim: int = 4
) -> Dict[str, Any]:
    """
    Dump all layer outputs from QCSNN24 RR→Both model.
    
    Args:
        model: Dictionary with {"net24": trunk, "net2": binary_head, "net4": multi_head}
        inputs: Input tensor (N, 1, 184) - 180 signal + 4 RR features
        num_steps: Number of timesteps
        out_dir: Output directory for dump files
        sig_len: Signal length (default 180)
        rr_dim: RR feature dimension (default 4)
    
    Returns:
        Dictionary with predictions and accumulated outputs
    """
    net24 = model["net24"]
    net2 = model["net2"]
    net4 = model["net4"]
    
    net24.eval()
    net2.eval()
    net4.eval()
    
    os.makedirs(out_dir, exist_ok=True)
    
    # Ensure input is (N, 1, L) format
    if inputs.ndim == 2:
        inputs = inputs.unsqueeze(1)
    
    N, _, total_len = inputs.shape
    assert total_len == sig_len + rr_dim, f"Expected {sig_len + rr_dim} features, got {total_len}"
    
    # Split signal and RR features
    x_sig = inputs[:, :, :sig_len]  # (N, 1, 180)
    rr = inputs[:, :, sig_len:].squeeze(1)  # (N, 4)
    
    print(f"Input shape: {inputs.shape}")
    print(f"Signal shape: {x_sig.shape}, RR shape: {rr.shape}")
    print(f"Running {num_steps} timesteps...")
    print(f"Output directory: {out_dir}")
    
    # Reset all LIF neurons
    utils.reset(net24)
    utils.reset(net2)
    utils.reset(net4)
    
    # Accumulators
    bin_acc = torch.zeros(N, 2, dtype=torch.float32, device=inputs.device)
    multi_acc = torch.zeros(N, 4, dtype=torch.float32, device=inputs.device)
    trunk_cache = []  # Cache trunk output for each timestep
    
    # ======================== STAGE-1: TRUNK + BINARY HEAD ========================
    for step in range(num_steps):
        should_write = (step == 0) or (step == num_steps - 1)
        step_suffix = f"_step{step:02d}" if should_write else ""
        
        print(f"\n=== Timestep {step}/{num_steps-1} ===")
        
        # ----- Input (signal only for trunk) -----
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(x_sig), f"{out_dir}/input{step_suffix}_ppt.txt")
        
        # ==================== TRUNK BLOCK 1 ====================
        # QIN1
        qin1 = net24.qcsnet24_cblk1_input(x_sig)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(qin1), f"{out_dir}/qin1{step_suffix}_ppt.txt")
        
        # Conv1
        conv1 = net24.qcsnet24_cblk1_qconv1d(qin1)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(conv1), f"{out_dir}/conv1{step_suffix}_ppt.txt")
        
        # BN1
        bn1 = net24.qcsnet24_cblk1_batch_norm(conv1)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(bn1), f"{out_dir}/bn1{step_suffix}_ppt.txt")
        
        # LIF1
        bn1_val = bn1.value if hasattr(bn1, "value") else bn1
        spk1 = net24.qcsnet24_cblk1_leaky(bn1_val)
        if isinstance(spk1, tuple):
            spk1 = spk1[0]
        if should_write:
            write_featmap_txt(spikes_to_binary(spk1), f"{out_dir}/lif1{step_suffix}_ppt.txt")
        
        # MaxPool1
        mp1 = net24.qcsnet24_cblk1_max_pool(spk1.float())
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(mp1), f"{out_dir}/mp1{step_suffix}_ppt.txt")
        
        # ==================== TRUNK BLOCK 2 ====================
        # QIN2
        qin2 = net24.qcsnet24_cblk2_input(mp1)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(qin2), f"{out_dir}/qi2{step_suffix}_ppt.txt")
        
        # Conv2
        conv2 = net24.qcsnet24_cblk2_qconv1d(qin2)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(conv2), f"{out_dir}/conv2{step_suffix}_ppt.txt")
        
        # BN2
        bn2 = net24.qcsnet24_cblk2_batch_norm(conv2)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(bn2), f"{out_dir}/bn2{step_suffix}_ppt.txt")
        
        # LIF2
        bn2_val = bn2.value if hasattr(bn2, "value") else bn2
        spk2 = net24.qcsnet24_cblk2_leaky(bn2_val)
        if isinstance(spk2, tuple):
            spk2 = spk2[0]
        if should_write:
            write_featmap_txt(spikes_to_binary(spk2), f"{out_dir}/lif2{step_suffix}_ppt.txt")
        
        # MaxPool2
        mp2 = net24.qcsnet24_cblk2_max_pool(spk2.float())
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(mp2), f"{out_dir}/mp2{step_suffix}_ppt.txt")
        
        # ==================== TRUNK BLOCK 3 ====================
        # QIN3
        qin3 = net24.qcsnet24_cblk3_input(mp2)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(qin3), f"{out_dir}/qi3{step_suffix}_ppt.txt")
        
        # Conv3
        conv3 = net24.qcsnet24_cblk3_qconv1d(qin3)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(conv3), f"{out_dir}/conv3{step_suffix}_ppt.txt")
        
        # BN3
        bn3 = net24.qcsnet24_cblk3_batch_norm(conv3)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(bn3), f"{out_dir}/bn3{step_suffix}_ppt.txt")
        
        # LIF3
        bn3_val = bn3.value if hasattr(bn3, "value") else bn3
        spk3 = net24.qcsnet24_cblk3_leaky(bn3_val)
        if isinstance(spk3, tuple):
            spk3 = spk3[0]
        if should_write:
            write_featmap_txt(spikes_to_binary(spk3), f"{out_dir}/lif3{step_suffix}_ppt.txt")
        
        # MaxPool3 (trunk output: 24×20 = 480)
        mp3 = net24.qcsnet24_cblk3_max_pool(spk3.float())
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(mp3), f"{out_dir}/mp3{step_suffix}_ppt.txt")
        
        # Flatten trunk output
        trunk_flat = net24.qcsnet24_flatten(mp3)  # (N, 480)
        
        # Cache trunk output for stage-2
        trunk_cache.append(trunk_flat.clone())
        
        # ==================== BINARY HEAD ====================
        # Binary head input: trunk (480) + RR (4) = 484
        bin_input = torch.cat([trunk_flat, rr], dim=1)  # (N, 484)
        
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(bin_input.unsqueeze(-1)), 
                            f"{out_dir}/bin_input{step_suffix}_ppt.txt")
        
        # BIN_QI
        bin_qi = net2.qcsnet2_lblk1_input(bin_input)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(prepare_for_write(bin_qi)), 
                            f"{out_dir}/bin_qi{step_suffix}_ppt.txt")
        
        # BIN_FC
        bin_fc = net2.qcsnet2_lblk1_qlinear(bin_qi)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(prepare_for_write(bin_fc)), 
                            f"{out_dir}/bin_fc{step_suffix}_ppt.txt")
        
        # BIN_LIF
        bin_fc_val = bin_fc.value if hasattr(bin_fc, "value") else bin_fc
        bin_out = net2.qcsnet2_lblk1_leaky(bin_fc_val)
        if isinstance(bin_out, tuple):
            bin_spk, bin_mem = bin_out
        else:
            bin_spk = bin_out
        
        if should_write:
            write_featmap_txt(spikes_to_binary(prepare_for_write(bin_spk)), 
                            f"{out_dir}/bin_lif{step_suffix}_ppt.txt")
        
        # Accumulate binary head spikes
        bin_spk_t = bin_spk.value if hasattr(bin_spk, 'value') else bin_spk
        bin_spk_flat = bin_spk_t.view(N, 2) if bin_spk_t.ndim != 2 else bin_spk_t
        bin_acc += (bin_spk_flat > 0).float()
        
        print(f"  Binary head: spk=[{bin_spk_flat[0, 0].item():.0f}, {bin_spk_flat[0, 1].item():.0f}]"
              f"  acc=[{bin_acc[0, 0].item():.0f}, {bin_acc[0, 1].item():.0f}]")
    
    # Gate decision: pred2 = 1 if sum_abn > sum_norm
    pred2 = (bin_acc[:, 1] > bin_acc[:, 0]).long()
    print(f"\n=== Stage-1 Complete ===")
    print(f"Binary accumulated: [{bin_acc[0, 0].item():.0f}, {bin_acc[0, 1].item():.0f}]")
    print(f"Gate decision: pred2={pred2[0].item()} (0=Normal, 1=Abnormal→route to Stage-2)")
    
    # ======================== STAGE-2: MULTI HEAD ========================
    # Reset multi head LIFs (not trunk or binary head)
    utils.reset(net4)
    
    print(f"\n=== Stage-2: Multi Head ===")
    
    for step in range(num_steps):
        should_write = (step == 0) or (step == num_steps - 1)
        step_suffix = f"_step{step:02d}" if should_write else ""
        
        # Get cached trunk output for this timestep
        trunk_flat = trunk_cache[step]  # (N, 480)
        
        # Multi head input: trunk (480) + RR (4) = 484
        multi_input = torch.cat([trunk_flat, rr], dim=1)  # (N, 484)
        
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(multi_input.unsqueeze(-1)), 
                            f"{out_dir}/multi_input{step_suffix}_ppt.txt")
        
        # ==================== MULTI HEAD LAYER 1 ====================
        # MULTI_QI1
        multi_qi1 = net4.qcsnet4_lblk1_input(multi_input)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(prepare_for_write(multi_qi1)), 
                            f"{out_dir}/multi_qi1{step_suffix}_ppt.txt")
        
        # MULTI_FC1 (484 → 128)
        multi_fc1 = net4.qcsnet4_lblk1_qlinear(multi_qi1)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(prepare_for_write(multi_fc1)), 
                            f"{out_dir}/multi_fc1{step_suffix}_ppt.txt")
        
        # MULTI_LIF1
        multi_fc1_val = multi_fc1.value if hasattr(multi_fc1, "value") else multi_fc1
        multi_spk1 = net4.qcsnet4_lblk1_leaky(multi_fc1_val)
        if isinstance(multi_spk1, tuple):
            multi_spk1 = multi_spk1[0]
        
        if should_write:
            write_featmap_txt(spikes_to_binary(prepare_for_write(multi_spk1)), 
                            f"{out_dir}/multi_lif1{step_suffix}_ppt.txt")
        
        # ==================== MULTI HEAD LAYER 2 ====================
        # MULTI_QI2
        multi_spk1_t = multi_spk1.value if hasattr(multi_spk1, 'value') else multi_spk1
        multi_qi2 = net4.qcsnet4_lblk2_input(multi_spk1_t.float())
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(prepare_for_write(multi_qi2)), 
                            f"{out_dir}/multi_qi2{step_suffix}_ppt.txt")
        
        # MULTI_FC2 (128 → 4)
        multi_fc2 = net4.qcsnet4_lblk2_qlinear(multi_qi2)
        if should_write:
            write_featmap_txt(qt_to_int8_ch_first(prepare_for_write(multi_fc2)), 
                            f"{out_dir}/multi_fc2{step_suffix}_ppt.txt")
        
        # MULTI_LIF2
        multi_fc2_val = multi_fc2.value if hasattr(multi_fc2, "value") else multi_fc2
        multi_out = net4.qcsnet4_lblk2_leaky(multi_fc2_val)
        if isinstance(multi_out, tuple):
            multi_spk2, multi_mem2 = multi_out
        else:
            multi_spk2 = multi_out
        
        if should_write:
            write_featmap_txt(spikes_to_binary(prepare_for_write(multi_spk2)), 
                            f"{out_dir}/multi_lif2{step_suffix}_ppt.txt")
        
        # Accumulate multi head spikes
        multi_spk2_t = multi_spk2.value if hasattr(multi_spk2, 'value') else multi_spk2
        multi_spk_flat = multi_spk2_t.view(N, 4) if multi_spk2_t.ndim != 2 else multi_spk2_t
        multi_acc += (multi_spk_flat > 0).float()
        
        if step == 0 or step == num_steps - 1:
            print(f"  Step {step}: multi_acc=[{multi_acc[0, 0].item():.0f}, {multi_acc[0, 1].item():.0f}, "
                  f"{multi_acc[0, 2].item():.0f}, {multi_acc[0, 3].item():.0f}]")
    
    # Stage-2 prediction: argmax
    pred4 = multi_acc.argmax(dim=1)
    
    # Final prediction: use pred2 gate
    final_pred = torch.where(pred2 == 0, torch.zeros_like(pred4), pred4)
    
    # Write accumulated outputs
    write_featmap_txt(bin_acc.unsqueeze(-1).to(torch.int8), 
                     f"{out_dir}/bin_accumulated_ppt.txt")
    write_featmap_txt(multi_acc.unsqueeze(-1).to(torch.int8), 
                     f"{out_dir}/multi_accumulated_ppt.txt")
    
    print(f"\n=== Final Results ===")
    print(f"Binary accumulated: [{bin_acc[0, 0].item():.0f}, {bin_acc[0, 1].item():.0f}]")
    print(f"Multi accumulated: [{multi_acc[0, 0].item():.0f}, {multi_acc[0, 1].item():.0f}, "
          f"{multi_acc[0, 2].item():.0f}, {multi_acc[0, 3].item():.0f}]")
    print(f"pred2={pred2[0].item()}, pred4={pred4[0].item()}, final={final_pred[0].item()}")
    
    return {
        'pred2': pred2,
        'pred4': pred4,
        'final_pred': final_pred,
        'bin_acc': bin_acc,
        'multi_acc': multi_acc,
    }

In [7]:
# ============================================================================
# Batch processing for multiple samples
# ============================================================================
@torch.no_grad()
def dump_qcsnn24_rrboth_batch(
    model: Dict,
    dataloader,
    num_steps: int = 10,
    out_dir: str = "./compare_outputs/",
    max_samples: int = None,
    sig_len: int = 180,
    rr_dim: int = 4
) -> Dict[str, Any]:
    """
    Dump layer outputs for multiple samples from a dataloader.
    Each sample gets its own set of files with sample index in filename.
    """
    net24 = model["net24"]
    net2 = model["net2"]
    net4 = model["net4"]
    
    net24.eval()
    net2.eval()
    net4.eval()
    
    os.makedirs(out_dir, exist_ok=True)
    
    all_results = []
    sample_count = 0
    
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        if max_samples and sample_count >= max_samples:
            break
        
        # Process one sample at a time for detailed dumps
        for i in range(inputs.shape[0]):
            if max_samples and sample_count >= max_samples:
                break
            
            sample = inputs[i:i+1]  # Keep batch dimension
            label = labels[i].item()
            
            print(f"\n{'='*60}")
            print(f"Sample {sample_count + 1}, Label={label}")
            print(f"{'='*60}")
            
            # Create sample-specific output directory
            sample_dir = f"{out_dir}/sample_{sample_count + 1:04d}/"
            
            result = dump_qcsnn24_rrboth(
                model=model,
                inputs=sample,
                num_steps=num_steps,
                out_dir=sample_dir,
                sig_len=sig_len,
                rr_dim=rr_dim
            )
            
            result['label'] = label
            result['sample_idx'] = sample_count
            all_results.append(result)
            
            sample_count += 1
    
    print(f"\n{'='*60}")
    print(f"Dumped {sample_count} samples to {out_dir}")
    print(f"{'='*60}")
    
    return all_results


# ============================================================================
# Example usage
# ============================================================================
if __name__ == "__main__":
    print("""
Usage example:

    from dump_brevitas_qcsnn24_rrboth import dump_qcsnn24_rrboth
    
    # Create/load your model
    model24 = create_qcsnn_model24_rr_both(...)
    # Load checkpoint...
    
    # Get a sample (shape: N, 1, 184)
    sample = torch.randn(1, 1, 184)  # or from your dataloader
    
    # Dump all layers
    results = dump_qcsnn24_rrboth(
        model=model24,
        inputs=sample,
        num_steps=10,
        out_dir="./compare_outputs/"
    )
    
    # Compare with C++ outputs:
    # python compare_layers.py --dirs ./compare_outputs/ ./cpp_outputs/ \\
    #     --layers input conv1 bn1 lif1 mp1 qi2 conv2 bn2 lif2 mp2 \\
    #              qi3 conv3 bn3 lif3 mp3 bin_input bin_qi bin_fc bin_lif \\
    #              multi_input multi_qi1 multi_fc1 multi_lif1 \\
    #              multi_qi2 multi_fc2 multi_lif2
    """)


Usage example:

    from dump_brevitas_qcsnn24_rrboth import dump_qcsnn24_rrboth
    
    # Create/load your model
    model24 = create_qcsnn_model24_rr_both(...)
    # Load checkpoint...
    
    # Get a sample (shape: N, 1, 184)
    sample = torch.randn(1, 1, 184)  # or from your dataloader
    
    # Dump all layers
    results = dump_qcsnn24_rrboth(
        model=model24,
        inputs=sample,
        num_steps=10,
        out_dir="./compare_outputs/"
    )
    
    # Compare with C++ outputs:
    # python compare_layers.py --dirs ./compare_outputs/ ./cpp_outputs/ \
    #     --layers input conv1 bn1 lif1 mp1 qi2 conv2 bn2 lif2 mp2 \
    #              qi3 conv3 bn3 lif3 mp3 bin_input bin_qi bin_fc bin_lif \
    #              multi_input multi_qi1 multi_fc1 multi_lif1 \
    #              multi_qi2 multi_fc2 multi_lif2
    


In [22]:
_, _, rr_stats = load_train_test_data(return_rr_stats=True)

Loaded: /home/velox-217533/Projects/fau_projects/research/data/mitbih_processed_intra_patient_4class_180_center90_rr_features_filtered/train  X=(80557, 184)
Beat(180) stats: min=-4.87 max=6.05 mean=0.0000 std=1.0000
RR(4) raw stats : min=-5.11 max=53.83 mean=0.6510 std=0.4938
RR(4) norm stats: min=-22.69 max=115.29 mean=0.0000 std=1.0000
Class distribution: Counter({0: 72073, 2: 5616, 1: 2224, 3: 644})
X shape: (80557, 1, 184) | y shape: (80557,)


In [28]:

# Load test data (adjust function name if different)
mitbih_test_path='/home/velox-217533/Projects/fau_projects/research/data/mitbih_processed_intra_patient_4class_180_center90_rr_features_filtered/test_one'

test_data, test_targets = load_train_test_data(mitbih_path=mitbih_test_path, 
                                               rr_stats=rr_stats, 
                                               return_rr_stats=False)
test_dataset = create_csnn_datasets(test_data, test_targets)

print(f"Test set size: {len(test_data)}")
print(f"Test normal samples: {(test_targets == 0).sum()}")
print(f"Test sveb samples: {(test_targets == 1).sum()}")
print(f"Test veb samples: {(test_targets == 2).sum()}")
print(f"Test f samples: {(test_targets == 3).sum()}")

Loaded: /home/velox-217533/Projects/fau_projects/research/data/mitbih_processed_intra_patient_4class_180_center90_rr_features_filtered/test_one  X=(1, 184)
Beat(180) stats: min=-1.76 max=4.01 mean=-0.0000 std=1.0000
RR(4) raw stats : min=-0.62 max=1.10 mean=0.3437 std=0.6185
RR(4) norm stats: min=-2.78 max=1.44 mean=-1.0076 std=1.5315
Class distribution: Counter({1: 1})
X shape: (1, 1, 184) | y shape: (1,)
train_dataset created successfully
Test set size: 1
Test normal samples: 0
Test sveb samples: 1
Test veb samples: 0
Test f samples: 0


In [29]:
print(rr_stats["mean"])

[0.77589595 0.7736819  1.0521744  0.00221263]


In [30]:
print(rr_stats["std"])

[0.22838424 0.22442298 0.45782378 0.22528282]


In [11]:
checkpoint_path="fold3_with_stage12_rr_features_SMOTE_fulltrain_finetuned_20260210_165114.pth"
# 1. Load model from checkpoint file
model24 = load_qcsnn24_dict_state(
    model_factory=create_qcsnn_model24_rr_both,
    checkpoint_path=checkpoint_path,
    device="cpu"
)



Output sizes: Block1=89, Block2=43, Block3=20
Flattened size: 480
Stage 1 input: 484 (trunk + RR)
Stage 2 input: 484 (trunk + RR)
[Loaded] Model from fold3_with_stage12_rr_features_SMOTE_fulltrain_finetuned_20260210_165114.pth -> cpu


In [ ]:


import os
import math
import numpy as np
import torch

import brevitas.nn as qnn
import snntorch as snn

# -----------------------
# Utilities
# -----------------------

def _ensure_dir(d):
    os.makedirs(d, exist_ok=True)

def _to_one(x):
    if isinstance(x, torch.Tensor):
        if x.numel() == 1:
            return float(x.detach().cpu().item())
        return x.detach().cpu().numpy()
    if isinstance(x, (float, int)):
        return x
    return x

def _get_bit_scale_zp_from_quant(q):
    """Return (bit_width, scale, zero_point, signed) from a brevitas quantizer-like object."""
    bit_width = None
    scale = None
    zero_point = None
    signed = True
    # bit width
    for k in ('bit_width', 'bit_width_impl', 'bit_width_f'):
        v = getattr(q, k, None)
        if v is None: continue
        try:
            v = v() if callable(v) else v
            bit_width = int(_to_one(v))
            break
        except Exception:
            pass
    # scale
    for k in ('scale', 'tensor_scale', 'scale_impl', 'act_scale', 'weight_scale'):
        v = getattr(q, k, None)
        if v is None: continue
        try:
            v = v() if callable(v) else v
            v = _to_one(v)
            if isinstance(v, (float, int)):
                scale = float(v)
                break
            if isinstance(v, np.ndarray) and v.size == 1:
                scale = float(v.item())
                break
        except Exception:
            pass
    # zero-point
    for k in ('zero_point', 'zero_point_impl', 'zp'):
        v = getattr(q, k, None)
        if v is None: continue
        try:
            v = v() if callable(v) else v
            zero_point = int(round(_to_one(v)))
            break
        except Exception:
            pass
    # signed
    sattr = getattr(q, 'signed', None)
    if isinstance(sattr, bool):
        signed = sattr
    elif zero_point is None:
        signed = True
    # defaults
    if bit_width is None: bit_width = 8
    if scale is None:     scale = 1.0
    if zero_point is None: zero_point = 0
    return bit_width, float(scale), int(zero_point), bool(signed)

def _quantize_multiplier(real_multiplier: float):
    """TFLite-style integer multiplier/shift approximation for a positive real multiplier."""
    if real_multiplier <= 0.0:
        return 0, 0
    mantissa, exponent = math.frexp(real_multiplier)  # real = mantissa * 2^exponent, mantissa in [0.5,1)
    q = int(round(mantissa * (1 << 31)))
    if q == (1 << 31):
        q //= 2
        exponent += 1
    shift = 31 - exponent
    if shift < 0:
        q <<= (-shift)
        shift = 0
    return int(q), int(shift)

# helper types used in headers (names only; arrays use ap_int in C++)
def _sum_weights_per_out(W_int8: np.ndarray) -> np.ndarray:
    # W_int8 shape: [OUT_CH, IN_CH, K]
    # returns int32 sums per OUT_CH
    return W_int8.reshape(W_int8.shape[0], -1).sum(axis=1).astype(np.int32)

def _bias_int32_vector(b_f: np.ndarray, s_in: float, s_w: float, out_ch: int) -> np.ndarray:
    # Quantize bias or return zeros if no bias provided
    if b_f is None:
        return np.zeros(out_ch, dtype=np.int32)
    s_bias = s_in * s_w if (s_in and s_w) else 1.0
    return np.round(b_f / s_bias).astype(np.int32)


def _qt_weight_int8_per_tensor(W_f: np.ndarray, scale: float, zero_point: int = 0):
    Wq = np.round(W_f / scale) + zero_point
    return np.clip(Wq, -128, 127).astype(np.int8)

def _bias_int32_from_float(b_f: np.ndarray, s_in: float, s_w: float):
    s_bias = s_in * s_w
    if s_bias == 0.0: s_bias = 1.0
    bq = np.round(b_f / s_bias).astype(np.int32)
    return bq

def _as1(x):
    if isinstance(x, (list, tuple)):
        return int(x[0])
    return int(x)

def _guard_out_scale(name: str, out_scale: float):
    if out_scale is None or not np.isfinite(out_scale) or out_scale <= 0.0:
        raise ValueError(f"{name}: invalid out_scale={out_scale}")

def _guard_in_scale(name: str, in_scale: float):
    if in_scale is None or not np.isfinite(in_scale) or in_scale <= 0.0:
        raise ValueError(f"{name}: invalid in_scale={in_scale}")

def _guard_weight_scale(name: str, w_scale: float):
    if w_scale is None or not np.isfinite(w_scale) or w_scale <= 0.0:
        raise ValueError(f"{name}: invalid weight_scale={w_scale}")

def _id_guard_macro(base: str):
    return base.upper().replace('/', '_').replace('.', '_')

def _sym(name: str):
    """C identifier from layer name (keep as-is but safe for C)."""
    return name.replace('/', '_').replace('.', '_')

def _fmt_int_list(vals, per_line=16):
    out = []
    line = []
    for i, v in enumerate(vals):
        line.append(str(int(v)))
        if (i + 1) % per_line == 0:
            out.append(", ".join(line))
            line = []
    if line:
        out.append(", ".join(line))
    return ",\n    ".join(out)

def _fmt_array_2d(arr2d):
    rows = []
    for r in arr2d:
        rows.append("{ " + _fmt_int_list(r) + " }")
    return "{\n  " + ",\n  ".join(rows) + "\n}"

def _fmt_array_3d(arr3d):
    blocks = []
    for b in arr3d:
        blocks.append(_fmt_array_2d(b))
    return "{\n" + ",\n".join(blocks) + "\n}"

# -----------------------
# Emitters
# -----------------------

def _emit_header_open(fp, guard, ns="hls4csnn1d_cblk_sd"):
    fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
    fp.write("#include <ap_int.h>\n\n")
    fp.write("#include \"../constants24_sd.h\"\n\n")  # ✅ important
    fp.write(f"namespace {ns} {{\n\n")

def _emit_header_close(fp, guard, ns="hls4csnn1d_cblk_sd"):
    fp.write(f"}} // namespace\n#endif // {guard}\n")

def _emit_conv1d_header(path, lname, W_int8, rq_mult, rq_shift,
                        bias_int32_vec, input_zp, weight_sum_vec, m):
    # Guard: QCSNET2_CBLK1_QCONV1D_WEIGHTS_H style
    guard = _id_guard_macro(f"{_sym(lname)}_WEIGHTS_H")
    with open(path, "w") as fp:
        _emit_header_open(fp, guard)  # writes includes + namespace line

        sym   = _sym(lname)
        OC, IC, K = W_int8.shape
        stride = _as1(m.stride)

        # Structural constants (optional but handy for TBs)
        fp.write(f"const int {sym}_OUT_CH = {OC};\n")
        fp.write(f"const int {sym}_IN_CH  = {IC};\n")
        fp.write(f"const int {sym}_KERNEL_SIZE = {K};\n")
        fp.write(f"const int {sym}_STRIDE = {stride};\n\n")

        # Input ZP (INT8)
        fp.write(f"const ap_int<8> {sym}_input_zero_point = {int(input_zp)};\n\n")

        # Requantization arrays (duplicated per OUT_CH to match your API)
        fp.write(f"const ap_int<32> {sym}_scale_multiplier[{OC}] = {{\n  ")
        fp.write(_fmt_int_list([rq_mult]*OC))
        fp.write("\n};\n\n")

        fp.write(f"const int {sym}_right_shift[{OC}] = {{\n  ")
        fp.write(_fmt_int_list([rq_shift]*OC))
        fp.write("\n};\n\n")

        # Bias (INT32)
        fp.write(f"const acc32_t {sym}_bias[{OC}] = {{\n  ")
        fp.write(_fmt_int_list(bias_int32_vec))
        fp.write("\n};\n\n")

        # Weight sums for asymmetric correction (INT32)
        fp.write(f"const acc32_t {sym}_weight_sum[{OC}] = {{\n  ")
        fp.write(_fmt_int_list(weight_sum_vec))
        fp.write("\n};\n\n")

        # Weights (INT8): [OUT_CH][IN_CH][K]
        fp.write(f"const ap_int<8> {sym}_weights[{OC}][{IC}][{K}] = ")
        fp.write(_fmt_array_3d(W_int8))
        fp.write(";\n\n")

        _emit_header_close(fp, guard)  # closes namespace + guard


def _emit_linear_header(path, lname,
                           W_int8,                 # [OUT][IN] int8
                           rq_mult, rq_shift,      # per-tensor constants, repeated per OUT
                           bias_int32_vec,         # [OUT] int32
                           input_zp,               # int (will be placed as ap_int<8>)
                           weight_sum_vec):        # [OUT] int32
    guard = _id_guard_macro(f"{_sym(lname)}_WEIGHTS_H")
    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("#include <hls_stream.h>\n#include <ap_int.h>\n#include \"../constants24_sd.h\"\n\n")
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")

        sym = _sym(lname)
        OUT, IN = W_int8.shape

        # Structural (optional helpers)
        fp.write(f"const int {sym}_OUTPUT_SIZE = {OUT};\n")
        fp.write(f"const int {sym}_INPUT_SIZE  = {IN};\n\n")

        # Input zero-point (matches template type)
        fp.write(f"const ap_int<8> {sym}_input_zero_point = {int(input_zp)};\n\n")

        # Requant arrays (match template types; repeat the per-tensor constants)
        fp.write(f"const ap_int<32> {sym}_scale_multiplier[{OUT}] = {{\n  ")
        fp.write(_fmt_int_list([rq_mult] * OUT))
        fp.write("\n};\n\n")

        fp.write(f"const int {sym}_right_shift[{OUT}] = {{\n  ")
        fp.write(_fmt_int_list([rq_shift] * OUT))
        fp.write("\n};\n\n")

        # Bias and weight_sum (acc domain)
        fp.write(f"using acc32_t = ap_int<32>;\n")
        fp.write(f"const acc32_t {sym}_bias[{OUT}] = {{\n  ")
        fp.write(_fmt_int_list(bias_int32_vec))
        fp.write("\n};\n\n")

        fp.write(f"const acc32_t {sym}_weight_sum[{OUT}] = {{\n  ")
        fp.write(_fmt_int_list(weight_sum_vec))
        fp.write("\n};\n\n")

        # Weights (use ap_int8_c to match your template)
        fp.write(f"const ap_int8_c {sym}_weights[{OUT}][{IN}] = ")
        fp.write(_fmt_array_2d(W_int8))
        fp.write(";\n\n")

        fp.write("} // namespace hls4csnn1d_cblk_sd\n")
        fp.write(f"#endif // {guard}\n")



def _emit_bn_header(path, lname, w_q, b32, mult_arr, shift_arr):
    guard = _id_guard_macro(f"{_sym(lname)}_BN_H")
    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("#include <hls_stream.h>\n#include <ap_int.h>\n#include \"../constants24_sd.h\"\n\n")
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")
        sym = _sym(lname); C = len(w_q)

        fp.write(f"const int {sym}_C = {C};\n\n")

        fp.write(f"const ap_int8_c {sym}_weight[{C}] = {{\n  {_fmt_int_list(w_q)}\n}};\n\n")
        fp.write(f"const ap_int<32> {sym}_bias[{C}] = {{\n  {_fmt_int_list(b32)}\n}};\n\n")
        fp.write(f"const ap_int<32> {sym}_scale_multiplier[{C}] = {{\n  {_fmt_int_list(mult_arr)}\n}};\n\n")
        fp.write(f"const int {sym}_right_shift[{C}] = {{\n  {_fmt_int_list(shift_arr)}\n}};\n\n")

        fp.write("} // namespace hls4csnn1d_cblk_sd\n")
        fp.write(f"#endif // {guard}\n")



def _emit_lif_header_scalar_sd(path, lname, beta_q, theta_q, scale_q, frac_bits):
    guard = _id_guard_macro(f"{_sym(lname)}_LIF_H")
    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("#include <hls_stream.h>\n#include <ap_int.h>\n#include \"../constants24_sd.h\"\n\n")
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")
        sym = _sym(lname)
        fp.write(f"enum {{ {sym}_FRAC_BITS = {int(frac_bits)} }};\n")
        fp.write(f"const ap_int<16> {sym}_beta_int   = {int(beta_q)};\n")
        fp.write(f"const ap_int<16> {sym}_theta_int  = {int(theta_q)};\n")
        fp.write(f"const ap_int<16> {sym}_scale_int  = {int(scale_q)};\n\n")
        fp.write("} // namespace hls4csnn1d_cblk_sd\n")
        fp.write(f"#endif // {guard}\n")


def _emit_lif_header_vector_sd(path, lname, beta_arr_q, theta_arr_q, scale_q, frac_bits):
    guard = _id_guard_macro(f"{_sym(lname)}_LIF_H")
    sym   = _sym(lname)
    N     = len(beta_arr_q)
    if len(theta_arr_q) != N:
        raise ValueError("beta/theta array lengths must match")

    def _fmt_list(vals, per_line=16):
        rows = []
        for i in range(0, len(vals), per_line):
            chunk = ", ".join(str(int(v)) for v in vals[i:i+per_line])
            rows.append("    " + chunk)
        return "{\n" + ",\n".join(rows) + "\n}"

    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("#include <hls_stream.h>\n#include <ap_int.h>\n#include \"../constants24_sd.h\"\n\n")
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")
        fp.write(f"enum {{ {sym}_FRAC_BITS = {int(frac_bits)}, {sym}_OUT_CH = {int(N)} }};\n")
        fp.write(f"const ap_int<16> {sym}_scale_int = {int(scale_q)};\n\n")
        fp.write(f"const ap_int<16> {sym}_beta_int[{sym}_OUT_CH] = "  + _fmt_list(beta_arr_q)  + ";\n\n")
        fp.write(f"const ap_int<16> {sym}_theta_int[{sym}_OUT_CH] = " + _fmt_list(theta_arr_q) + ";\n\n")
        fp.write("} // namespace hls4csnn1d_cblk_sd\n")
        fp.write(f"#endif // {guard}\n")



INT16_MIN, INT16_MAX = -32768, 32767

def _to_q_i16(x: float, Q: int) -> int:
    return int(np.clip(round(float(x) * Q), INT16_MIN, INT16_MAX))

def _tensor_to_q_i16_list(t: torch.Tensor, Q: int):
    flat = t.detach().float().reshape(-1).cpu().tolist()
    return [_to_q_i16(v, Q) for v in flat]


_Q_SCALE = 1 << 12   # e.g., FRAC_BITS = 12

def _emit_qparams_header(path, lname, bit_w, scale, zp):
    guard = _id_guard_macro(f"QPARAMS_{_sym(lname)}_H")
    sym = _sym(lname)
    sym_base = sym  # no renaming, no suffix stripping

    # Q-encode the activation scale for HLS QuantIdentity (uses _Q_SCALE; not emitted)
    act_scale_int = _to_q_i16(float(scale), _Q_SCALE)

    with open(path, "w") as fp:
        _emit_header_open(fp, guard)  # must include <ap_int.h> and open your namespace

        fp.write("// Activation quantization parameters (optional for kernels)\n")
        fp.write(f"const int   {sym}_bit_width = {int(bit_w)};\n")
        fp.write(f"// const float {sym}_scale     = {float(scale):.10g};  // kept for reference only\n")
        fp.write(f"const ap_int<16> {sym_base}_act_scale_int = {int(act_scale_int)};\n")
        fp.write(f"const int   {sym}_zero_point= {int(zp)};\n\n")

        _emit_header_close(fp, guard)  # close namespace and guard


# Add this function to your weight extraction code (after the other _emit_* functions)

def _emit_rr_stats_header(path, rr_stats, input_scale=None):
    """
    Emit header with RR normalization statistics from training data.
    
    Args:
        path: Output header file path
        rr_stats: Dict with "mean" and "std" arrays (each length 4)
        input_scale: Optional input quantization scale (for reference)
    """
    guard = "RR_NORMALIZATION_STATS_H"
    
    rr_mean = rr_stats["mean"]
    rr_std = rr_stats["std"]
    
    # Precompute 1/std for faster runtime (multiply instead of divide)
    rr_std_inv = 1.0 / np.maximum(rr_std, 1e-8)
    
    with open(path, "w") as fp:
        fp.write(f"#ifndef {guard}\n#define {guard}\n\n")
        fp.write("/**\n")
        fp.write(" * RR Feature Normalization Statistics\n")
        fp.write(" * \n")
        fp.write(" * These values are computed from the TRAINING set and must be used\n")
        fp.write(" * for both training and test/inference to avoid data leakage.\n")
        fp.write(" * \n")
        fp.write(" * Normalization formula:\n")
        fp.write(" *   rr_normalized[i] = (rr_raw[i] - RR_MEAN[i]) / RR_STD[i]\n")
        fp.write(" *                    = (rr_raw[i] - RR_MEAN[i]) * RR_STD_INV[i]\n")
        fp.write(" * \n")
        fp.write(" * Input layout (184 features):\n")
        fp.write(" *   [0-179]:   ECG beat samples (already z-scored per beat in CSV)\n")
        fp.write(" *   [180-183]: RR interval features (need normalization)\n")
        fp.write(" */\n\n")
        
        fp.write("namespace hls4csnn1d_cblk_sd {\n\n")
        
        fp.write("// RR feature indices\n")
        fp.write("static const int RR_FEATURE_START = 180;\n")
        fp.write("static const int RR_FEATURE_COUNT = 4;\n\n")
        
        # Mean values
        fp.write("// Training set mean for each RR feature\n")
        fp.write("static const float RR_MEAN[4] = {\n")
        fp.write(f"    {rr_mean[0]:.10f}f,   // RR_prev\n")
        fp.write(f"    {rr_mean[1]:.10f}f,   // RR_next\n")
        fp.write(f"    {rr_mean[2]:.10f}f,   // RR_ratio\n")
        fp.write(f"    {rr_mean[3]:.10f}f    // RR_diff\n")
        fp.write("};\n\n")
        
        # Std values
        fp.write("// Training set std for each RR feature\n")
        fp.write("static const float RR_STD[4] = {\n")
        fp.write(f"    {rr_std[0]:.10f}f,   // RR_prev\n")
        fp.write(f"    {rr_std[1]:.10f}f,   // RR_next\n")
        fp.write(f"    {rr_std[2]:.10f}f,   // RR_ratio\n")
        fp.write(f"    {rr_std[3]:.10f}f    // RR_diff\n")
        fp.write("};\n\n")
        
        # Precomputed 1/std for faster runtime
        fp.write("// Precomputed 1/std for faster computation (multiply instead of divide)\n")
        fp.write("static const float RR_STD_INV[4] = {\n")
        fp.write(f"    {rr_std_inv[0]:.10f}f,   // 1/RR_prev_std\n")
        fp.write(f"    {rr_std_inv[1]:.10f}f,   // 1/RR_next_std\n")
        fp.write(f"    {rr_std_inv[2]:.10f}f,   // 1/RR_ratio_std\n")
        fp.write(f"    {rr_std_inv[3]:.10f}f    // 1/RR_diff_std\n")
        fp.write("};\n\n")
        
        # Input scale (if provided)
        if input_scale is not None:
            fp.write(f"// Input quantization scale (from QuantIdentity)\n")
            fp.write(f"static const float RR_INPUT_SCALE = {float(input_scale):.10f}f;\n")
            fp.write(f"static const float RR_INPUT_SCALE_INV = {1.0/float(input_scale):.10f}f;\n\n")
        
        fp.write("} // namespace hls4csnn1d_cblk_sd\n\n")
        fp.write(f"#endif // {guard}\n")
    
    print(f"[emit] RR stats header written to: {path}")


# -----------------------
# Orchestrator
# -----------------------

def emit_headers_for_model(model: torch.nn.Module,
                           example_input: torch.Tensor,
                           out_dir: str = "headers_int",
                           lif_frac_bits: int = 12):
    model.eval()
    _ensure_dir(out_dir)

    # Run one dry forward to initialize any lazy buffers (ignore output tuples)
    with torch.no_grad():
        try:
            _ = model(example_input)
        except Exception:
            pass

    # Track current activation qparams (propagated as in your graph)
    current_act = {"bit_width": 8, "scale": 1.0, "zero_point": 0}
    
    last_out_ch = None  # initialize outside the loop

    for name, m in model.named_modules():
        if m is model:
            continue

        # QuantIdentity (export activation qparams as optional header)
        if isinstance(m, qnn.QuantIdentity):
            aq = getattr(m, 'act_quant', getattr(m, 'output_quant', None))
            bit_w, s, zp, _ = _get_bit_scale_zp_from_quant(aq)
            _emit_qparams_header(os.path.join(out_dir, f"qparams_{name}.h"), name, bit_w, s, zp)
            current_act = {"bit_width": bit_w, "scale": s, "zero_point": zp}
            continue


        if isinstance(m, qnn.QuantConv1d):
            # Float → INT8 weights
            Wf = m.weight.detach().cpu().numpy()            # [OUT, IN, K]
            wq = getattr(m, 'weight_quant', None)
            wb, s_w, z_w, _ = _get_bit_scale_zp_from_quant(wq)
        
            _guard_in_scale(name, current_act['scale'])
            _guard_weight_scale(name, s_w)
        
            W_int8 = _qt_weight_int8_per_tensor(Wf, s_w, z_w)
        
            # Output activation qparams (for requant)
            oq = getattr(m, 'output_quant', None)
            ob, s_out, z_out, _ = _get_bit_scale_zp_from_quant(oq)
            _guard_out_scale(name, s_out)
        
            # Integer requant constants (per-tensor → repeat per OUT_CH)
            M = (current_act['scale'] * s_w) / s_out
            rq_mult, rq_shift = _quantize_multiplier(M)
        
            # Bias (if present) → INT32 vector (length OUT_CH); else zeros
            b_f = m.bias.detach().cpu().numpy() if (hasattr(m, 'bias') and m.bias is not None) else None
            bias_int32_vec = _bias_int32_vector(b_f, current_act['scale'], s_w, W_int8.shape[0])
        
            # Weight sums per output channel (for asymmetric correction)
            weight_sum_vec = _sum_weights_per_out(W_int8)
        
            # Input zero point (INT8) for asymmetric correction
            input_zp = current_act['zero_point']
        
            # Emit header that matches your Conv1D_SD::forward signature
            _emit_conv1d_header(
                os.path.join(out_dir, f"{name}_weights.h"),
                name,
                W_int8,
                rq_mult, rq_shift,
                bias_int32_vec,
                input_zp,
                weight_sum_vec,
                m
            )
        
            # Advance activation qparams for the next layer
            current_act = {"bit_width": ob, "scale": s_out, "zero_point": z_out}
            continue


        # BN -> ScaleBias (for BatchNorm1dToQuantScaleBias)
        if hasattr(qnn, 'BatchNorm1dToQuantScaleBias') and isinstance(m, qnn.BatchNorm1dToQuantScaleBias):
            # gamma, beta (float, per channel)
            gamma = _to_one(getattr(m, 'weight', None)) if getattr(m, 'weight', None) is not None else _to_one(getattr(m, 'scale', None))
            beta  = _to_one(getattr(m, 'bias',   None)) if getattr(m, 'bias',   None) is not None else _to_one(getattr(m, 'beta',  None))
            if gamma is None: gamma = 1.0
            if beta  is None: beta  = 0.0
            gamma = np.array(gamma, dtype=np.float32).reshape(-1)
            beta  = np.array(beta,  dtype=np.float32).reshape(-1)
            C = gamma.shape[0]
        
            # Input/output quant
            s_in = float(current_act['scale']);  z_in = int(current_act['zero_point']);  _guard_in_scale(name, s_in)
            oq   = getattr(m, 'output_quant', None)
            _, s_out, _, _ = _get_bit_scale_zp_from_quant(oq);  _guard_out_scale(name, s_out)
        
            # Brevitas BN quantized weight
            qW = m.quant_weight()  # IntQuantTensor
            w_q = qW.int().detach().cpu().numpy().astype(np.int8)    # [C]
            s_w = float(_to_one(qW.scale))                            # scalar
        
            # Shared requant scale S and its integer pair
            S = s_w * (s_in / s_out)
            mult_S, shift_S = _quantize_multiplier(S)
        
            # Int32 bias per channel (no int8 clipping)
            # M_c = (s_in/s_out) * gamma_c  = S * w_q[c]  (approximately)
            M = gamma * (s_in / s_out)                          # [C]
            b32 = np.round((beta / s_out - M * z_in) / S).astype(np.int32)  # [C]
        
            _emit_bn_header(
                os.path.join(out_dir, f"{name}_bn.h"),
                name,
                w_q.tolist(),
                b32.tolist(),
                [mult_S] * C,
                [shift_S] * C
            )
        
            # Advance activation qparams
            current_act = {"bit_width": current_act['bit_width'], "scale": s_out, "zero_point": 0}
            continue



        # QuantLinear
        if isinstance(m, qnn.QuantLinear):
            # Float → INT8 weights
            Wf = m.weight.detach().cpu().numpy()             # [OUT, IN]
            wq = getattr(m, 'weight_quant', None)
            wb, s_w, z_w, _ = _get_bit_scale_zp_from_quant(wq)
        
            _guard_in_scale(name, current_act['scale'])
            _guard_weight_scale(name, s_w)
        
            W_int8 = _qt_weight_int8_per_tensor(Wf, s_w, z_w)
        
            # Output activation qparams (for requant)
            oq = getattr(m, 'output_quant', None)
            ob, s_out, z_out, _ = _get_bit_scale_zp_from_quant(oq)
            _guard_out_scale(name, s_out)
        
            # Requant: M = (s_in * s_w) / s_out
            M = (current_act['scale'] * s_w) / s_out
            rq_mult, rq_shift = _quantize_multiplier(M)
        
            # Bias (if present) → INT32; else zeros
            b_f = m.bias.detach().cpu().numpy() if (hasattr(m, 'bias') and m.bias is not None) else None
            bias_int32_vec = _bias_int32_vector(b_f, current_act['scale'], s_w, W_int8.shape[0])
        
            # Weight sums for asymmetric correction: sum over input dim
            weight_sum_vec = W_int8.sum(axis=1).astype(np.int32)
        
            # Input zero-point for asymmetric correction
            input_zp = current_act['zero_point']
        
            # Emit header that matches Linear1D_SD::forward
            _emit_linear_header(
                os.path.join(out_dir, f"{name}_weights.h"),
                name,
                W_int8,
                rq_mult, rq_shift,
                bias_int32_vec,
                input_zp,
                weight_sum_vec
            )
        
            # Advance activation qparams
            current_act = {"bit_width": ob, "scale": s_out, "zero_point": z_out}
            continue

        
        if isinstance(m, snn.Leaky):
            scale_in = float(current_act['scale'])
            Q = 1 << lif_frac_bits
        
            # Grab beta/threshold as tensors (handles both scalar and vector)
            beta_t  = m.beta if isinstance(m.beta, torch.Tensor) else torch.as_tensor(m.beta)
            thr_t   = m.threshold if isinstance(m.threshold, torch.Tensor) else torch.as_tensor(m.threshold)
        
            beta_t  = beta_t.detach().float()
            print("beta: ", beta_t)
            thr_t   = thr_t.detach().float()
        
            # Number of “neurons” (channels) from parameter size
            n_beta  = int(beta_t.numel())
            n_thr   = int(thr_t.numel())
            if n_beta != n_thr and n_beta != 1 and n_thr != 1:
                raise ValueError(f"LIF param size mismatch: beta has {n_beta}, threshold has {n_thr}")
        
            # Broadcast if one is scalar
            out_ch = max(n_beta, n_thr)
            if n_beta == 1 and out_ch > 1:
                beta_t = beta_t.expand(out_ch)
            if n_thr == 1 and out_ch > 1:
                thr_t = thr_t.expand(out_ch)
        
            # Quantize
            beta_arr_q  = _tensor_to_q_i16_list(beta_t, Q)
            theta_arr_q = _tensor_to_q_i16_list(thr_t, Q)
            scale_q     = _to_q_i16(scale_in, Q)   # keep scale scalar for now
        
            # Emit vector or scalar header depending on out_ch
            out_path = os.path.join(out_dir, f"{name}_lif.h")
            if out_ch > 1:
                _emit_lif_header_vector_sd(
                    out_path, name, beta_arr_q, theta_arr_q, scale_q, lif_frac_bits
                )
            else:
                _emit_lif_header_scalar_sd(
                    out_path, name, beta_arr_q[0], theta_arr_q[0], scale_q, lif_frac_bits
                )
        
            # LIF outputs binary spikes {0,1} → treat next op input scale as 1.0
            current_act = {"bit_width": 8, "scale": 1.0, "zero_point": 0}
            continue
            
        # # =====================================================================
        # # NEW: Emit RR stats header if provided
        # # =====================================================================
        # if rr_stats is not None:
        #     _emit_rr_stats_header(
        #         os.path.join(out_dir, "rr_normalization_stats.h"),
        #         rr_stats,
        #     )

    print(f"[emit] C++ headers written to: {os.path.abspath(out_dir)}")


In [ ]:
def emit_headers_for_qcsnn24_dict(
    model_dict, 
    out_dir="smote_with_stage2_rr_features_headers_int24", 
    lif_frac_bits=12,
    trunk_size=480,
    rr_dim=4,
    rr_to_stage1=True,
    rr_to_stage2=True,
    rr_stats=None,
):
    """
    Export net24, net2, net4 into the SAME output folder.
    
    Ablation configurations:
      - Ablation 1: rr_to_stage1=True,  rr_to_stage2=False  (RR → S1 only)
      - Ablation 2: rr_to_stage1=False, rr_to_stage2=True   (RR → S2 only)
      - Ablation 3: rr_to_stage1=True,  rr_to_stage2=True   (RR → Both)
      - No RR:      rr_to_stage1=False, rr_to_stage2=False
    """
    _ensure_dir(out_dir)
    x_trunk = torch.randn(1, 1, 180)
    stage1_input = trunk_size + rr_dim if rr_to_stage1 else trunk_size
    x_head2 = torch.randn(1, stage1_input)
    stage2_input = trunk_size + rr_dim if rr_to_stage2 else trunk_size
    x_head4 = torch.randn(1, stage2_input)
    
    print(f"Exporting weights:")
    print(f"  net24 input: (1, 1, 180)")
    print(f"  net2 (Stage 1) input: (1, {stage1_input}) [RR: {rr_to_stage1}]")
    print(f"  net4 (Stage 2) input: (1, {stage2_input}) [RR: {rr_to_stage2}]")
    
    emit_headers_for_model(model_dict["net24"], x_trunk, out_dir=out_dir, lif_frac_bits=lif_frac_bits)
    emit_headers_for_model(model_dict["net2"],  x_head2, out_dir=out_dir, lif_frac_bits=lif_frac_bits)
    emit_headers_for_model(model_dict["net4"],  x_head4, out_dir=out_dir, lif_frac_bits=lif_frac_bits)
    
    if rr_stats is not None:
        _emit_rr_stats_header(os.path.join(out_dir, "rr_stats.h"), rr_stats)
    
    print(f"[emit] C++ headers written to: {os.path.abspath(out_dir)}")

In [ ]:
model_export = {
    "net24": model24["net24"].cpu().eval(),
    "net2":  model24["net2"].cpu().eval(),
    "net4":  model24["net4"].cpu().eval(),
}

# Ablation 1: RR → Stage 1 only (THE WINNER)
emit_headers_for_qcsnn24_dict(
    model_dict=model_export,
    out_dir="weights_sd/intra_patient/smote_rr_both_stages_v2",
    rr_to_stage1=True,
    rr_to_stage2=True,
    rr_stats=rr_stats  # <-- Pass your rr_stats here
)

In [31]:
# 2. Get a sample from your dataloader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    pin_memory=(device.type == "cuda"),
    num_workers=0,
)
sample, label = next(iter(test_loader))  # (1, 1, 184)



In [32]:
# 3. Dump all layers
results = dump_qcsnn24_rrboth(
    model=model24,
    inputs=sample,
    num_steps=10,
    out_dir="../../compare_fusedbody_outputs/"
)

# 4. Run C++ dumper on same data, then compare
# python compare_layers.py --dirs ./cpp_outputs/ ./py_outputs/ --layers conv1_step00 bn1_step00 ...

Input shape: torch.Size([1, 1, 184])
Signal shape: torch.Size([1, 1, 180]), RR shape: torch.Size([1, 4])
Running 10 timesteps...
Output directory: ../../compare_fusedbody_outputs/

=== Timestep 0/9 ===
  Binary head: spk=[0, 1]  acc=[0, 1]

=== Timestep 1/9 ===
  Binary head: spk=[0, 1]  acc=[0, 2]

=== Timestep 2/9 ===
  Binary head: spk=[0, 1]  acc=[0, 3]

=== Timestep 3/9 ===
  Binary head: spk=[0, 1]  acc=[0, 4]

=== Timestep 4/9 ===
  Binary head: spk=[0, 1]  acc=[0, 5]

=== Timestep 5/9 ===
  Binary head: spk=[0, 1]  acc=[0, 6]

=== Timestep 6/9 ===
  Binary head: spk=[0, 1]  acc=[0, 7]

=== Timestep 7/9 ===
  Binary head: spk=[0, 1]  acc=[0, 8]

=== Timestep 8/9 ===
  Binary head: spk=[0, 1]  acc=[0, 9]

=== Timestep 9/9 ===
  Binary head: spk=[0, 1]  acc=[0, 10]

=== Stage-1 Complete ===
Binary accumulated: [0, 10]
Gate decision: pred2=1 (0=Normal, 1=Abnormal→route to Stage-2)

=== Stage-2: Multi Head ===
  Step 0: multi_acc=[0, 1, 0, 0]
  Step 9: multi_acc=[0, 10, 0, 0]

=== F